# Global Automotive Investment Database — Block 2

## Global Security Master and listing-identity construction

This notebook loads the persisted Block 1 Parquet outputs. It does not execute
Block 1 and therefore does not re-download or reprocess SEC N-PORT quarters.

### Block 2 responsibilities

- construct deterministic provisional issuer and security IDs;
- separate issuer identity from listed-security identity;
- preserve identifier, name and listing histories with source provenance;
- resolve canonical exchange, MIC, listing country and share class;
- attach canonical IDs and listing metadata to point-in-time ETF intervals;
- surface collisions, ambiguity, unresolved identities and unresolved listings;
- create downstream interfaces for all regional fundamentals modules;
- persist Block 2 outputs for Block 3 and later modules.

### Listing-resolution policy

The listing resolver uses a controlled evidence hierarchy:

1. source-native exchange or MIC fields, where Block 1 provides them;
2. exact ticker suffixes or exchange prefixes;
3. country, currency, ISIN prefix and ticker-structure combinations;
4. an explicit unresolved status where the evidence is insufficient.

Issuer domicile alone never determines a listing venue. A Chinese issuer may
have separate Mainland, Hong Kong and US-listed securities, each with its own
`security_id`.

In [ ]:
# 1. INSTALL / IMPORT DEPENDENCIES
# ------------------------------------------------

from __future__ import annotations

import gc
import hashlib
import json
import re
import unicodedata

from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 240)

print("Dependencies imported.")

Dependencies imported.


In [ ]:
# 2. USER SETTINGS AND BLOCK 1 PARQUET INPUT
# ------------------------------------------------

USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

    PROJECT_ROOT = Path(
        "/content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy"
    )
else:
    PROJECT_ROOT = Path(
        "/content/global_automotive_investment_database"
    )

DATA_ROOT = PROJECT_ROOT / "data"

BLOCK_1_OUTPUT_DIR = (
    DATA_ROOT
    / "interim"
    / "block_1"
)

BLOCK_1_MANIFEST_PATH = (
    BLOCK_1_OUTPUT_DIR
    / "block_1_manifest.json"
)

BLOCK_2_OUTPUT_DIR = (
    DATA_ROOT
    / "interim"
    / "block_2"
)

BLOCK_2_MANIFEST_PATH = (
    BLOCK_2_OUTPUT_DIR
    / "block_2_manifest.json"
)

BLOCK_2_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ISSUER_ID_PREFIX = "GAI"
SECURITY_ID_PREFIX = "GAS"

# Incremented because listed-security identity and downstream contracts change.
SECURITY_MASTER_SCHEMA_VERSION = "1.2.0"

PERSIST_BLOCK_2_OUTPUTS = True
OVERWRITE_PERSISTED_OUTPUTS = True


def load_manifest_tables(
    manifest_path: Path,
    required_table_names: set[str] | None = None,
) -> tuple[dict[str, pd.DataFrame], dict]:
    """Load selected Parquet tables declared in a block manifest."""

    if not manifest_path.exists():
        raise FileNotFoundError(
            f"Required manifest not found: {manifest_path}\n"
            "Run revised Block 1 fully before running Block 2."
        )

    with manifest_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        manifest = json.load(file)

    records = {
        item["table_name"]: item
        for item in manifest.get(
            "tables",
            [],
        )
    }

    selected_names = (
        set(records)
        if required_table_names is None
        else set(required_table_names)
    )

    missing = selected_names.difference(
        records
    )

    if missing:
        raise RuntimeError(
            "Block 1 manifest is missing required tables: "
            f"{sorted(missing)}"
        )

    loaded = {}

    for table_name in sorted(
        selected_names
    ):
        table_path = Path(
            records[table_name][
                "path"
            ]
        )

        if not table_path.exists():
            raise FileNotFoundError(
                "Manifest entry exists but Parquet file is missing: "
                f"{table_path}"
            )

        loaded[table_name] = (
            pd.read_parquet(
                table_path
            )
        )

    return loaded, manifest


REQUIRED_BLOCK_1_TABLES = {
    "security_master_seed_df",
    "etf_constituent_intervals_df",
    "etf_constituent_snapshots_df",
    "etf_snapshot_index_df",
}

block_1_inputs, block_1_manifest = (
    load_manifest_tables(
        BLOCK_1_MANIFEST_PATH,
        REQUIRED_BLOCK_1_TABLES,
    )
)

print(
    "Loaded persisted Block 1 inputs without executing Block 1:"
)

for name, dataframe in (
    block_1_inputs.items()
):
    print(
        f"  {name}: "
        f"{len(dataframe):,} rows × "
        f"{len(dataframe.columns):,} columns"
    )

print(
    "\nBlock 1 manifest:",
    BLOCK_1_MANIFEST_PATH,
)

print(
    "Block 2 output directory:",
    BLOCK_2_OUTPUT_DIR,
)

Mounted at /content/drive
Loaded persisted Block 1 inputs without executing Block 1:
  etf_constituent_intervals_df: 8,028 rows × 28 columns
  etf_constituent_snapshots_df: 8,028 rows × 26 columns
  etf_snapshot_index_df: 107 rows × 10 columns
  security_master_seed_df: 8,028 rows × 15 columns

Block 1 manifest: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/interim/block_1/block_1_manifest.json
Block 2 output directory: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/interim/block_2


In [ ]:
# 3. INPUT CONTRACT VALIDATION
# ------------------------------------------------

BLOCK_1_COLUMN_ALIASES = {
    "security_title": "security_name",
    "issuer_lei": "lei",
    "investment_country": "country",
    "source_accession_number": "accession_number",

    # Optional listing aliases retained when Block 1 happens to contain them.
    "market": "exchange",
    "exchange_code": "exchange",
    "listing_exchange": "exchange",
    "exchange_mic": "mic",
    "market_identifier_code": "mic",
    "security_country": "listing_country",
    "listing_domicile": "listing_country",
    "share_type": "share_class",
}

REQUIRED_BLOCK_1_DATAFRAMES = {
    "security_master_seed_df": {
        "etf",
        "issuer_name",
        "ticker",
        "isin",
        "cusip",
        "issuer_lei",
        "investment_country",
        "currency",
        "snapshot_date",
        "available_date",
        "source_accession_number",
    },
    "etf_constituent_intervals_df": {
        "etf",
        "issuer_name",
        "ticker",
        "isin",
        "cusip",
        "issuer_lei",
        "snapshot_date",
        "available_date",
        "next_available_date",
        "accession_number",
    },
    "etf_constituent_snapshots_df": {
        "etf",
        "issuer_name",
        "ticker",
        "isin",
        "cusip",
        "issuer_lei",
        "snapshot_date",
        "filing_date",
        "available_date",
        "accession_number",
    },
    "etf_snapshot_index_df": {
        "etf",
        "snapshot_date",
        "filing_date",
        "available_date",
        "next_available_date",
        "accession_number",
    },
}


def validate_dataframe_contract(
    name: str,
    dataframe: pd.DataFrame,
    required_columns: Iterable[str],
) -> None:
    if not isinstance(
        dataframe,
        pd.DataFrame,
    ):
        raise TypeError(
            f"{name} must be a pandas DataFrame."
        )

    missing = sorted(
        set(required_columns)
        .difference(
            dataframe.columns
        )
    )

    if missing:
        raise ValueError(
            f"{name} is missing required columns: {missing}"
        )


def canonicalise_block_1_columns(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """
    Rename known source-native fields without overwriting canonical columns.
    """

    frame = dataframe.copy()

    rename_map = {
        source: target
        for source, target
        in BLOCK_1_COLUMN_ALIASES.items()
        if (
            source in frame.columns
            and target not in frame.columns
        )
    }

    return frame.rename(
        columns=rename_map
    )


validated_block_1_inputs = {}

for name, required_columns in (
    REQUIRED_BLOCK_1_DATAFRAMES.items()
):
    dataframe = block_1_inputs[
        name
    ]

    validate_dataframe_contract(
        name,
        dataframe,
        required_columns,
    )

    validated_block_1_inputs[
        name
    ] = canonicalise_block_1_columns(
        dataframe
    )

block_1_inputs = (
    validated_block_1_inputs
)

security_master_seed_df = (
    block_1_inputs[
        "security_master_seed_df"
    ]
)

etf_constituent_intervals_df = (
    block_1_inputs[
        "etf_constituent_intervals_df"
    ]
)

etf_constituent_snapshots_df = (
    block_1_inputs[
        "etf_constituent_snapshots_df"
    ]
)

etf_snapshot_index_df = (
    block_1_inputs[
        "etf_snapshot_index_df"
    ]
)

optional_listing_columns = sorted(
    set(
        [
            "exchange",
            "mic",
            "listing_country",
            "share_class",
        ]
    )
    .intersection(
        security_master_seed_df.columns
    )
)

print(
    "Persisted Block 1 input contract validated and canonicalised."
)

print(
    "Optional listing columns found in Block 1:",
    optional_listing_columns,
)

for key, value in (
    block_1_inputs.items()
):
    print(
        f"{key}: "
        f"{len(value):,} rows × "
        f"{len(value.columns):,} columns"
    )

Persisted Block 1 input contract validated and canonicalised.
Optional listing columns found in Block 1: []
security_master_seed_df: 8,028 rows × 15 columns
etf_constituent_intervals_df: 8,028 rows × 28 columns
etf_constituent_snapshots_df: 8,028 rows × 26 columns
etf_snapshot_index_df: 107 rows × 10 columns


In [ ]:
# 4. NORMALISATION, LISTING MAPS AND DETERMINISTIC ID HELPERS
# ------------------------------------------------

NULL_STRINGS = {
    "",
    "NAN",
    "NONE",
    "NULL",
    "N/A",
    "NA",
    "-",
    "--",
    "<NA>",
}


def clean_string(
    value: object,
) -> str | pd.NA:
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()

    if text.upper() in NULL_STRINGS:
        return pd.NA

    return text


def normalise_identifier(
    value: object,
) -> str | pd.NA:
    text = clean_string(
        value
    )

    if pd.isna(text):
        return pd.NA

    cleaned = re.sub(
        r"[^A-Za-z0-9]",
        "",
        str(text),
    ).upper()

    return (
        cleaned
        if cleaned
        else pd.NA
    )


def normalise_ticker(
    value: object,
) -> str | pd.NA:
    text = clean_string(
        value
    )

    if pd.isna(text):
        return pd.NA

    cleaned = re.sub(
        r"\s+",
        "",
        str(text),
    ).upper()

    return (
        cleaned
        if cleaned
        else pd.NA
    )


def normalise_name(
    value: object,
) -> str | pd.NA:
    text = clean_string(
        value
    )

    if pd.isna(text):
        return pd.NA

    ascii_text = (
        unicodedata
        .normalize(
            "NFKD",
            str(text),
        )
        .encode(
            "ascii",
            "ignore",
        )
        .decode()
    )

    ascii_text = ascii_text.upper()

    ascii_text = re.sub(
        r"&",
        " AND ",
        ascii_text,
    )

    ascii_text = re.sub(
        r"[^A-Z0-9]+",
        " ",
        ascii_text,
    )

    ascii_text = re.sub(
        r"\s+",
        " ",
        ascii_text,
    ).strip()

    return (
        ascii_text
        if ascii_text
        else pd.NA
    )


COUNTRY_ALIASES = {
    "CHINA": "CN",
    "MAINLAND CHINA": "CN",
    "PRC": "CN",
    "CHN": "CN",
    "PEOPLE'S REPUBLIC OF CHINA": "CN",
    "HONG KONG": "HK",
    "HONGKONG": "HK",
    "UNITED STATES": "US",
    "USA": "US",
    "UNITED KINGDOM": "GB",
    "GREAT BRITAIN": "GB",
    "SOUTH KOREA": "KR",
    "REPUBLIC OF KOREA": "KR",
    "KOREA": "KR",
    "JAPAN": "JP",
    "GERMANY": "DE",
    "FRANCE": "FR",
    "ITALY": "IT",
    "CANADA": "CA",
    "AUSTRALIA": "AU",
    "TAIWAN": "TW",
    "INDIA": "IN",
    "NETHERLANDS": "NL",
    "SWEDEN": "SE",
    "SWITZERLAND": "CH",
}


def normalise_country(
    value: object,
) -> str | pd.NA:
    text = clean_string(
        value
    )

    if pd.isna(text):
        return pd.NA

    upper = str(text).upper()

    return COUNTRY_ALIASES.get(
        upper,
        upper,
    )


def normalise_currency(
    value: object,
) -> str | pd.NA:
    text = normalise_identifier(
        value
    )

    return (
        text
        if pd.isna(text)
        else str(text)[:3]
    )


EXCHANGE_ALIASES = {
    # Mainland China
    "SH": "SSE",
    "SS": "SSE",
    "SSE": "SSE",
    "SHSE": "SSE",
    "XSHG": "SSE",
    "SHANGHAI": "SSE",
    "SHANGHAISTOCKEXCHANGE": "SSE",

    "SZ": "SZSE",
    "SZSE": "SZSE",
    "XSHE": "SZSE",
    "SHENZHEN": "SZSE",
    "SHENZHENSTOCKEXCHANGE": "SZSE",

    "BJ": "BSE",
    "BSE": "BSE",
    "XBEI": "BSE",
    "BEIJING": "BSE",
    "BEIJINGSTOCKEXCHANGE": "BSE",

    # Hong Kong
    "HK": "HKEX",
    "HKEX": "HKEX",
    "XHKG": "HKEX",
    "HONGKONG": "HKEX",

    # United States
    "NYSE": "NYSE",
    "XNYS": "NYSE",
    "NASDAQ": "NASDAQ",
    "XNAS": "NASDAQ",
    "NYSEARCA": "NYSE_ARCA",
    "ARCX": "NYSE_ARCA",

    # Japan
    "T": "TSE",
    "JP": "TSE",
    "TSE": "TSE",
    "XTKS": "TSE",
    "TOKYO": "TSE",

    # Korea
    "KS": "KRX",
    "KQ": "KOSDAQ",
    "KRX": "KRX",
    "XKRX": "KRX",
    "KOSDAQ": "KOSDAQ",
    "XKOS": "KOSDAQ",

    # Taiwan
    "TW": "TWSE",
    "TT": "TWSE",
    "TWSE": "TWSE",
    "XTAI": "TWSE",

    # Europe and other common markets
    "L": "LSE",
    "LN": "LSE",
    "LSE": "LSE",
    "XLON": "LSE",

    "DE": "XETRA",
    "GR": "XETRA",
    "XETRA": "XETRA",
    "XETR": "XETRA",

    "PA": "EURONEXT_PARIS",
    "FP": "EURONEXT_PARIS",
    "XPAR": "EURONEXT_PARIS",

    "MI": "BORSA_ITALIANA",
    "IM": "BORSA_ITALIANA",
    "XMIL": "BORSA_ITALIANA",

    "SW": "SIX",
    "SIX": "SIX",
    "XSWX": "SIX",

    "ST": "NASDAQ_STOCKHOLM",
    "XSTO": "NASDAQ_STOCKHOLM",

    "TO": "TSX",
    "CN": "TSX",
    "XTSE": "TSX",

    "AX": "ASX",
    "AU": "ASX",
    "ASX": "ASX",
    "XASX": "ASX",
}

EXCHANGE_TO_MIC = {
    "SSE": "XSHG",
    "SZSE": "XSHE",
    "BSE": "XBEI",
    "HKEX": "XHKG",
    "NYSE": "XNYS",
    "NASDAQ": "XNAS",
    "NYSE_ARCA": "ARCX",
    "TSE": "XTKS",
    "KRX": "XKRX",
    "KOSDAQ": "XKOS",
    "TWSE": "XTAI",
    "LSE": "XLON",
    "XETRA": "XETR",
    "EURONEXT_PARIS": "XPAR",
    "BORSA_ITALIANA": "XMIL",
    "SIX": "XSWX",
    "NASDAQ_STOCKHOLM": "XSTO",
    "TSX": "XTSE",
    "ASX": "XASX",
}

EXCHANGE_TO_LISTING_COUNTRY = {
    "SSE": "CN",
    "SZSE": "CN",
    "BSE": "CN",
    "HKEX": "HK",
    "NYSE": "US",
    "NASDAQ": "US",
    "NYSE_ARCA": "US",
    "TSE": "JP",
    "KRX": "KR",
    "KOSDAQ": "KR",
    "TWSE": "TW",
    "LSE": "GB",
    "XETRA": "DE",
    "EURONEXT_PARIS": "FR",
    "BORSA_ITALIANA": "IT",
    "SIX": "CH",
    "NASDAQ_STOCKHOLM": "SE",
    "TSX": "CA",
    "ASX": "AU",
}


def normalise_exchange(
    value: object,
) -> str | pd.NA:
    text = clean_string(
        value
    )

    if pd.isna(text):
        return pd.NA

    compact = re.sub(
        r"[\s_\-\.]",
        "",
        str(text).upper(),
    )

    return EXCHANGE_ALIASES.get(
        compact,
        pd.NA,
    )


def normalise_mic(
    value: object,
) -> str | pd.NA:
    text = normalise_identifier(
        value
    )

    if pd.isna(text):
        return pd.NA

    exchange = EXCHANGE_ALIASES.get(
        str(text),
        pd.NA,
    )

    if pd.notna(exchange):
        return EXCHANGE_TO_MIC.get(
            exchange,
            str(text),
        )

    return (
        str(text)
        if len(str(text)) == 4
        else pd.NA
    )


def stable_hash(
    parts: Iterable[object],
    length: int = 20,
) -> str:
    canonical = "|".join(
        ""
        if pd.isna(value)
        else str(value)
        for value in parts
    )

    return (
        hashlib.sha256(
            canonical.encode(
                "utf-8"
            )
        )
        .hexdigest()
        .upper()[:length]
    )


def make_internal_id(
    prefix: str,
    identity_type: str,
    identity_value: str,
) -> str:
    return (
        f"{prefix}_"
        f"{stable_hash([SECURITY_MASTER_SCHEMA_VERSION, identity_type, identity_value])}"
    )


def first_non_null(
    *values: object,
) -> object:
    for value in values:
        if (
            not pd.isna(value)
            and str(value).strip() != ""
        ):
            return value

    return pd.NA


def extract_isin_country(
    value: object,
) -> str | pd.NA:
    isin = normalise_identifier(
        value
    )

    if (
        pd.notna(isin)
        and re.fullmatch(
            r"[A-Z]{2}[A-Z0-9]{9}\d",
            str(isin),
        )
    ):
        return str(isin)[:2]

    return pd.NA


def extract_numeric_ticker(
    value: object,
) -> str | pd.NA:
    ticker = normalise_ticker(
        value
    )

    if pd.isna(ticker):
        return pd.NA

    match = re.fullmatch(
        r"0*(\d{4,6})",
        str(ticker),
    )

    if not match:
        return pd.NA

    digits = match.group(1)

    return digits.zfill(
        len(str(ticker))
    )


def infer_mainland_exchange_from_code(
    code: object,
) -> str | pd.NA:
    if pd.isna(code):
        return pd.NA

    text = str(code).zfill(6)

    if text.startswith(
        (
            "600",
            "601",
            "603",
            "605",
            "688",
            "689",
            "900",
        )
    ):
        return "SSE"

    if text.startswith(
        (
            "000",
            "001",
            "002",
            "003",
            "200",
            "300",
            "301",
        )
    ):
        return "SZSE"

    if text.startswith(
        (
            "4",
            "8",
            "9",
        )
    ):
        return "BSE"

    return pd.NA


def classify_share_class(
    ticker: object,
    exchange: object,
) -> str | pd.NA:
    exchange_value = (
        str(exchange)
        if pd.notna(exchange)
        else ""
    )

    numeric = extract_numeric_ticker(
        ticker
    )

    if exchange_value == "SSE":
        if (
            pd.notna(numeric)
            and str(numeric).zfill(6).startswith("900")
        ):
            return "B_SHARE"

        return "A_SHARE"

    if exchange_value == "SZSE":
        if (
            pd.notna(numeric)
            and str(numeric).zfill(6).startswith("200")
        ):
            return "B_SHARE"

        return "A_SHARE"

    if exchange_value == "BSE":
        return "A_SHARE"

    if exchange_value == "HKEX":
        return "ORDINARY_OR_H_SHARE"

    if exchange_value in {
        "NYSE",
        "NASDAQ",
        "NYSE_ARCA",
    }:
        return "ORDINARY_OR_DEPOSITARY_RECEIPT"

    return pd.NA

In [ ]:
# 5. STANDARDISE THE BLOCK 1 SECURITY SEED
# ------------------------------------------------

DATE_COLUMNS = [
    "snapshot_date",
    "filing_date",
    "available_date",
    "next_available_date",
    "effective_date",
]

CANONICAL_OPTIONAL_COLUMNS = [
    "security_name",
    "issuer_name",
    "ticker",
    "isin",
    "cusip",
    "lei",
    "country",
    "currency",
    "exchange",
    "mic",
    "listing_country",
    "share_class",
    "series_name",
    "series_id",
    "holding_id",
    "accession_number",
    "filing_date",
    "snapshot_date",
    "available_date",
    "next_available_date",
    "effective_date",
]


def standardise_security_seed(
    seed: pd.DataFrame,
) -> pd.DataFrame:
    dataframe = (
        canonicalise_block_1_columns(
            seed
        )
    )

    for column in (
        CANONICAL_OPTIONAL_COLUMNS
    ):
        if column not in dataframe.columns:
            dataframe[column] = pd.NA

    for column in DATE_COLUMNS:
        dataframe[column] = pd.to_datetime(
            dataframe[column],
            errors="coerce",
        )

    for column in [
        "security_name",
        "issuer_name",
        "series_name",
        "accession_number",
        "series_id",
        "holding_id",
    ]:
        dataframe[column] = (
            dataframe[column]
            .map(
                clean_string
            )
        )

    dataframe[
        "security_name_normalised"
    ] = dataframe[
        "security_name"
    ].map(
        normalise_name
    )

    dataframe[
        "issuer_name_normalised"
    ] = dataframe[
        "issuer_name"
    ].map(
        normalise_name
    )

    dataframe[
        "ticker_normalised"
    ] = dataframe[
        "ticker"
    ].map(
        normalise_ticker
    )

    dataframe[
        "isin_normalised"
    ] = dataframe[
        "isin"
    ].map(
        normalise_identifier
    )

    dataframe[
        "cusip_normalised"
    ] = dataframe[
        "cusip"
    ].map(
        normalise_identifier
    )

    dataframe[
        "lei_normalised"
    ] = dataframe[
        "lei"
    ].map(
        normalise_identifier
    )

    dataframe[
        "issuer_country_normalised"
    ] = dataframe[
        "country"
    ].map(
        normalise_country
    )

    dataframe[
        "currency_normalised"
    ] = dataframe[
        "currency"
    ].map(
        normalise_currency
    )

    dataframe[
        "source_exchange_normalised"
    ] = dataframe[
        "exchange"
    ].map(
        normalise_exchange
    )

    dataframe[
        "source_mic_normalised"
    ] = dataframe[
        "mic"
    ].map(
        normalise_mic
    )

    dataframe[
        "source_listing_country_normalised"
    ] = dataframe[
        "listing_country"
    ].map(
        normalise_country
    )

    dataframe[
        "source_share_class"
    ] = dataframe[
        "share_class"
    ].map(
        clean_string
    )

    dataframe[
        "isin_country"
    ] = dataframe[
        "isin_normalised"
    ].map(
        extract_isin_country
    )

    dataframe[
        "source_system"
    ] = "SEC_NPORT"

    dataframe[
        "source_module"
    ] = "BLOCK_1_ETF_HOLDINGS"

    dataframe[
        "security_master_schema_version"
    ] = SECURITY_MASTER_SCHEMA_VERSION

    return dataframe


security_seed_standardised_df = (
    standardise_security_seed(
        security_master_seed_df
    )
)

print(
    "Standardised security observations:",
    len(
        security_seed_standardised_df
    ),
)

display(
    security_seed_standardised_df.head()
)

Standardised security observations: 8028


,etf,snapshot_date,available_date,issuer_name,security_name,ticker,isin,cusip,lei,other_identifier,other_identifier_description,country,currency,source_system,accession_number,exchange,mic,listing_country,share_class,series_name,series_id,holding_id,filing_date,next_available_date,effective_date,security_name_normalised,issuer_name_normalised,ticker_normalised,isin_normalised,cusip_normalised,lei_normalised,issuer_country_normalised,currency_normalised,source_exchange_normalised,source_mic_normalised,source_listing_country_normalised,source_share_class,isin_country,source_module,security_master_schema_version
0,CARZ,2019-09-30,2019-11-19,Aston Martin Lagonda Global Ho,Aston Martin Lagonda Global Holdings PLC,AML,GB00BFXZC448,000000000,213800167WOVOK5ZC776,<NA>,<NA>,GB,GBP,SEC_NPORT,0001752724-19-167449,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,NaT,NaT,ASTON MARTIN LAGONDA GLOBAL HOLDINGS PLC,ASTON MARTIN LAGONDA GLOBAL HO,AML,GB00BFXZC448,000000000,213800167WOVOK5ZC776,GB,GBP,<NA>,<NA>,<NA>,<NA>,GB,BLOCK_1_ETF_HOLDINGS,1.2.0
1,CARZ,2019-09-30,2019-11-19,BAIC Motor Corp Ltd,BAIC Motor Corp Ltd,1958,CNE100001TJ4,000000000,5299003EYHISIDNLZQ19,<NA>,<NA>,CN,HKD,SEC_NPORT,0001752724-19-167449,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,NaT,NaT,BAIC MOTOR CORP LTD,BAIC MOTOR CORP LTD,1958,CNE100001TJ4,000000000,5299003EYHISIDNLZQ19,CN,HKD,<NA>,<NA>,<NA>,<NA>,CN,BLOCK_1_ETF_HOLDINGS,1.2.0
2,CARZ,2019-09-30,2019-11-19,BNP PARIBAS SECURITIES CORP.,BNP PARIBAS SECURITIES CORP.,<NA>,<NA>,000000000,RCNB6OTYUAMMP879YW96,290076RBNP235,INTERNAL,US,USD,SEC_NPORT,0001752724-19-167449,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,NaT,NaT,BNP PARIBAS SECURITIES CORP,BNP PARIBAS SECURITIES CORP,<NA>,<NA>,000000000,RCNB6OTYUAMMP879YW96,US,USD,<NA>,<NA>,<NA>,<NA>,<NA>,BLOCK_1_ETF_HOLDINGS,1.2.0
3,CARZ,2019-09-30,2019-11-19,BRILLIANCE CHI,Brilliance China Automotive Holdings Ltd,1114,BMG1368B1028,000000000,5299005WPCSQN14NBM26,<NA>,<NA>,HK,HKD,SEC_NPORT,0001752724-19-167449,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,NaT,NaT,BRILLIANCE CHINA AUTOMOTIVE HOLDINGS LTD,BRILLIANCE CHI,1114,BMG1368B1028,000000000,5299005WPCSQN14NBM26,HK,HKD,<NA>,<NA>,<NA>,<NA>,BM,BLOCK_1_ETF_HOLDINGS,1.2.0
4,CARZ,2019-09-30,2019-11-19,BYD Co Ltd,BYD Co Ltd,1211,CNE100000296,000000000,5299005557VL7ULJ7A69,<NA>,<NA>,CN,HKD,SEC_NPORT,0001752724-19-167449,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,NaT,NaT,BYD CO LTD,BYD CO LTD,1211,CNE100000296,000000000,5299005557VL7ULJ7A69,CN,HKD,<NA>,<NA>,<NA>,<NA>,CN,BLOCK_1_ETF_HOLDINGS,1.2.0


In [ ]:
# 6. IDENTITY KEYS AND PROVISIONAL INTERNAL IDS
# ------------------------------------------------
#
# Issuer identity priority:
#   1. LEI
#   2. normalised issuer name + issuer country
#   3. normalised issuer name
#
# Security identity priority:
#   1. ISIN
#   2. CUSIP
#   3. issuer identity + ticker + currency
#   4. issuer identity + security name + currency


def assign_identity_keys(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    frame = dataframe.copy()

    def issuer_key(
        row: pd.Series,
    ) -> tuple[str, str]:
        if pd.notna(
            row[
                "lei_normalised"
            ]
        ):
            return (
                "LEI",
                str(
                    row[
                        "lei_normalised"
                    ]
                ),
            )

        name = row[
            "issuer_name_normalised"
        ]

        country = row[
            "issuer_country_normalised"
        ]

        if (
            pd.notna(name)
            and pd.notna(country)
        ):
            return (
                "NAME_COUNTRY",
                f"{name}|{country}",
            )

        if pd.notna(name):
            return (
                "NAME",
                str(name),
            )

        fallback = stable_hash(
            [
                row.get(
                    "accession_number"
                ),
                row.get(
                    "holding_id"
                ),
                row.get(
                    "ticker_normalised"
                ),
                row.get(
                    "currency_normalised"
                ),
            ]
        )

        return (
            "SOURCE_ROW",
            fallback,
        )

    def security_key(
        row: pd.Series,
    ) -> tuple[str, str]:
        if pd.notna(
            row[
                "isin_normalised"
            ]
        ):
            return (
                "ISIN",
                str(
                    row[
                        "isin_normalised"
                    ]
                ),
            )

        if pd.notna(
            row[
                "cusip_normalised"
            ]
        ):
            return (
                "CUSIP",
                str(
                    row[
                        "cusip_normalised"
                    ]
                ),
            )

        issuer_identity = row[
            "issuer_identity_value"
        ]

        ticker = row[
            "ticker_normalised"
        ]

        currency = row[
            "currency_normalised"
        ]

        if pd.notna(ticker):
            return (
                "ISSUER_TICKER_CURRENCY",
                f"{issuer_identity}|{ticker}|{currency}",
            )

        security_name = (
            first_non_null(
                row.get(
                    "security_name_normalised"
                ),
                row.get(
                    "issuer_name_normalised"
                ),
            )
        )

        return (
            "ISSUER_SECURITY_NAME",
            f"{issuer_identity}|{security_name}|{currency}",
        )

    issuer_pairs = frame.apply(
        issuer_key,
        axis=1,
        result_type="expand",
    )

    issuer_pairs.columns = [
        "issuer_identity_type",
        "issuer_identity_value",
    ]

    frame = pd.concat(
        [
            frame,
            issuer_pairs,
        ],
        axis=1,
    )

    security_pairs = frame.apply(
        security_key,
        axis=1,
        result_type="expand",
    )

    security_pairs.columns = [
        "security_identity_type",
        "security_identity_value",
    ]

    frame = pd.concat(
        [
            frame,
            security_pairs,
        ],
        axis=1,
    )

    frame[
        "issuer_id"
    ] = frame.apply(
        lambda row: make_internal_id(
            ISSUER_ID_PREFIX,
            row[
                "issuer_identity_type"
            ],
            row[
                "issuer_identity_value"
            ],
        ),
        axis=1,
    )

    frame[
        "security_id"
    ] = frame.apply(
        lambda row: make_internal_id(
            SECURITY_ID_PREFIX,
            row[
                "security_identity_type"
            ],
            row[
                "security_identity_value"
            ],
        ),
        axis=1,
    )

    return frame


security_seed_identified_df = (
    assign_identity_keys(
        security_seed_standardised_df
    )
)

display(
    security_seed_identified_df[
        [
            "issuer_name",
            "ticker",
            "isin",
            "cusip",
            "lei",
            "issuer_id",
            "security_id",
            "security_identity_type",
        ]
    ].head()
)

,issuer_name,ticker,isin,cusip,lei,issuer_id,security_id,security_identity_type
0,Aston Martin Lagonda Global Ho,AML,GB00BFXZC448,000000000,213800167WOVOK5ZC776,GAI_5CD5347816CF9C46E768,GAS_01F546D04C0C4F72A262,ISIN
1,BAIC Motor Corp Ltd,1958,CNE100001TJ4,000000000,5299003EYHISIDNLZQ19,GAI_342913484F832986D3FC,GAS_F6F7D7DB70A9FA35B66C,ISIN
2,BNP PARIBAS SECURITIES CORP.,<NA>,<NA>,000000000,RCNB6OTYUAMMP879YW96,GAI_7EE7791B88E1F75B6F48,GAS_4DBF216BD6CB838C4327,CUSIP
3,BRILLIANCE CHI,1114,BMG1368B1028,000000000,5299005WPCSQN14NBM26,GAI_3A233B7635E27B018082,GAS_8C2ADF81F06C1F2A1087,ISIN
4,BYD Co Ltd,1211,CNE100000296,000000000,5299005557VL7ULJ7A69,GAI_F9BF134E7CCD96A4C594,GAS_7BEEA4BAD5D4252C9825,ISIN


In [ ]:
# 7. RESOLVE LISTING IDENTITY AND BUILD CORE MASTER TABLES
# ------------------------------------------------


def mode_or_first(
    series: pd.Series,
) -> object:
    values = series.dropna()

    if values.empty:
        return pd.NA

    mode = values.mode()

    return (
        mode.iloc[0]
        if not mode.empty
        else values.iloc[0]
    )


TICKER_SUFFIX_RULES = [
    (r"^(?P<code>\d{6})\.(SS|SH)$", "SSE", 0.99),
    (r"^(?P<code>\d{6})\.(SZ)$", "SZSE", 0.99),
    (r"^(?P<code>\d{6})\.(BJ)$", "BSE", 0.99),
    (r"^(?P<code>\d{4,5})\.(HK)$", "HKEX", 0.99),
    (r"^(?P<code>\d{4})\.(T|JP)$", "TSE", 0.99),
    (r"^(?P<code>\d{6})\.(KS)$", "KRX", 0.99),
    (r"^(?P<code>\d{6})\.(KQ)$", "KOSDAQ", 0.99),
    (r"^(?P<code>\d{4})\.(TW|TT)$", "TWSE", 0.99),
    (r"^(?P<code>[A-Z0-9]+)\.(L|LN)$", "LSE", 0.98),
    (r"^(?P<code>[A-Z0-9]+)\.(DE|GR)$", "XETRA", 0.98),
    (r"^(?P<code>[A-Z0-9]+)\.(PA|FP)$", "EURONEXT_PARIS", 0.98),
    (r"^(?P<code>[A-Z0-9]+)\.(MI|IM)$", "BORSA_ITALIANA", 0.98),
    (r"^(?P<code>[A-Z0-9]+)\.(SW)$", "SIX", 0.98),
    (r"^(?P<code>[A-Z0-9]+)\.(ST)$", "NASDAQ_STOCKHOLM", 0.98),
    (r"^(?P<code>[A-Z0-9]+)\.(TO|CN)$", "TSX", 0.98),
    (r"^(?P<code>[A-Z0-9]+)\.(AX|AU)$", "ASX", 0.98),
]


def exact_ticker_suffix_resolution(
    ticker: object,
) -> tuple[object, object, float]:
    value = normalise_ticker(
        ticker
    )

    if pd.isna(value):
        return (
            pd.NA,
            pd.NA,
            np.nan,
        )

    for pattern, exchange, confidence in (
        TICKER_SUFFIX_RULES
    ):
        match = re.fullmatch(
            pattern,
            str(value),
        )

        if match:
            return (
                exchange,
                match.group(
                    "code"
                ),
                confidence,
            )

    return (
        pd.NA,
        pd.NA,
        np.nan,
    )


def infer_listing_from_observation(
    row: pd.Series,
) -> pd.Series:
    source_exchange = row.get(
        "source_exchange_normalised"
    )

    source_mic = row.get(
        "source_mic_normalised"
    )

    ticker = row.get(
        "ticker_normalised"
    )

    issuer_country = row.get(
        "issuer_country_normalised"
    )

    listing_country_source = row.get(
        "source_listing_country_normalised"
    )

    currency = row.get(
        "currency_normalised"
    )

    isin_country = row.get(
        "isin_country"
    )

    # 1. Source-native exchange.
    if pd.notna(source_exchange):
        exchange = str(
            source_exchange
        )

        return pd.Series({
            "exchange": exchange,
            "mic": (
                source_mic
                if pd.notna(source_mic)
                else EXCHANGE_TO_MIC.get(
                    exchange,
                    pd.NA,
                )
            ),
            "listing_country": (
                listing_country_source
                if pd.notna(listing_country_source)
                else EXCHANGE_TO_LISTING_COUNTRY.get(
                    exchange,
                    pd.NA,
                )
            ),
            "listing_ticker": ticker,
            "listing_resolution_method": "SOURCE_EXCHANGE",
            "listing_resolution_confidence": 1.00,
        })

    # 2. Source-native MIC.
    if pd.notna(source_mic):
        mic = str(source_mic)

        exchange = normalise_exchange(
            mic
        )

        return pd.Series({
            "exchange": exchange,
            "mic": mic,
            "listing_country": (
                listing_country_source
                if pd.notna(listing_country_source)
                else EXCHANGE_TO_LISTING_COUNTRY.get(
                    exchange,
                    pd.NA,
                )
            ),
            "listing_ticker": ticker,
            "listing_resolution_method": "SOURCE_MIC",
            "listing_resolution_confidence": 1.00,
        })

    # 3. Exact ticker suffix.
    suffix_exchange, suffix_code, suffix_confidence = (
        exact_ticker_suffix_resolution(
            ticker
        )
    )

    if pd.notna(suffix_exchange):
        exchange = str(
            suffix_exchange
        )

        return pd.Series({
            "exchange": exchange,
            "mic": EXCHANGE_TO_MIC.get(
                exchange,
                pd.NA,
            ),
            "listing_country": EXCHANGE_TO_LISTING_COUNTRY.get(
                exchange,
                pd.NA,
            ),
            "listing_ticker": (
                suffix_code
                if pd.notna(suffix_code)
                else ticker
            ),
            "listing_resolution_method": "EXACT_TICKER_SUFFIX",
            "listing_resolution_confidence": suffix_confidence,
        })

    numeric_ticker = extract_numeric_ticker(
        ticker
    )

    # 4. Country / ISIN / currency constrained rules.
    #
    # Mainland China:
    # - issuer country CN or ISIN prefix CN;
    # - six-digit numeric ticker;
    # - code family identifies SSE, SZSE or BSE.
    mainland_context = (
        (
            pd.notna(issuer_country)
            and str(issuer_country) == "CN"
        )
        or (
            pd.notna(isin_country)
            and str(isin_country) == "CN"
        )
    )

    if (
        mainland_context
        and pd.notna(numeric_ticker)
        and len(str(numeric_ticker)) == 6
        and str(numeric_ticker) != "000000"
    ):
        exchange = (
            infer_mainland_exchange_from_code(
                numeric_ticker
            )
        )

        if pd.notna(exchange):
            exchange = str(exchange)

            return pd.Series({
                "exchange": exchange,
                "mic": EXCHANGE_TO_MIC.get(
                    exchange,
                    pd.NA,
                ),
                "listing_country": "CN",
                "listing_ticker": str(
                    numeric_ticker
                ).zfill(6),
                "listing_resolution_method": "CN_CONTEXT_NUMERIC_TICKER",
                "listing_resolution_confidence": 0.93,
            })

    # Hong Kong:
    # - HKD currency or listing-country source HK;
    # - four/five-digit numeric ticker.
    hong_kong_context = (
        (
            pd.notna(currency)
            and str(currency) == "HKD"
        )
        or (
            pd.notna(listing_country_source)
            and str(listing_country_source) == "HK"
        )
    )

    if (
        hong_kong_context
        and pd.notna(numeric_ticker)
        and 4 <= len(str(numeric_ticker)) <= 5
    ):
        return pd.Series({
            "exchange": "HKEX",
            "mic": "XHKG",
            "listing_country": "HK",
            "listing_ticker": str(
                numeric_ticker
            ).zfill(5),
            "listing_resolution_method": "HKD_CONTEXT_NUMERIC_TICKER",
            "listing_resolution_confidence": 0.92,
        })

    # Japan.
    if (
        (
            pd.notna(issuer_country)
            and str(issuer_country) == "JP"
        )
        or (
            pd.notna(isin_country)
            and str(isin_country) == "JP"
        )
    ) and (
        pd.notna(numeric_ticker)
        and len(str(numeric_ticker)) == 4
    ):
        return pd.Series({
            "exchange": "TSE",
            "mic": "XTKS",
            "listing_country": "JP",
            "listing_ticker": str(
                numeric_ticker
            ),
            "listing_resolution_method": "JP_CONTEXT_NUMERIC_TICKER",
            "listing_resolution_confidence": 0.92,
        })

    # Korea.
    if (
        (
            pd.notna(issuer_country)
            and str(issuer_country) == "KR"
        )
        or (
            pd.notna(isin_country)
            and str(isin_country) == "KR"
        )
    ) and (
        pd.notna(numeric_ticker)
        and len(str(numeric_ticker)) == 6
    ):
        return pd.Series({
            "exchange": "KRX",
            "mic": "XKRX",
            "listing_country": "KR",
            "listing_ticker": str(
                numeric_ticker
            ).zfill(6),
            "listing_resolution_method": "KR_CONTEXT_NUMERIC_TICKER",
            "listing_resolution_confidence": 0.90,
        })

    # Taiwan.
    if (
        (
            pd.notna(issuer_country)
            and str(issuer_country) == "TW"
        )
        or (
            pd.notna(isin_country)
            and str(isin_country) == "TW"
        )
    ) and (
        pd.notna(numeric_ticker)
        and len(str(numeric_ticker)) == 4
    ):
        return pd.Series({
            "exchange": "TWSE",
            "mic": "XTAI",
            "listing_country": "TW",
            "listing_ticker": str(
                numeric_ticker
            ),
            "listing_resolution_method": "TW_CONTEXT_NUMERIC_TICKER",
            "listing_resolution_confidence": 0.90,
        })

    # Australia.
    if (
        (
            pd.notna(issuer_country)
            and str(issuer_country) == "AU"
        )
        or (
            pd.notna(isin_country)
            and str(isin_country) == "AU"
        )
    ) and (
        pd.notna(ticker)
        and re.fullmatch(
            r"[A-Z0-9]{2,6}",
            str(ticker),
        )
    ):
        return pd.Series({
            "exchange": "ASX",
            "mic": "XASX",
            "listing_country": "AU",
            "listing_ticker": ticker,
            "listing_resolution_method": "AU_CONTEXT_TICKER",
            "listing_resolution_confidence": 0.85,
        })

    # Canada.
    if (
        (
            pd.notna(issuer_country)
            and str(issuer_country) == "CA"
        )
        or (
            pd.notna(isin_country)
            and str(isin_country) == "CA"
        )
    ) and (
        pd.notna(currency)
        and str(currency) == "CAD"
    ):
        return pd.Series({
            "exchange": "TSX",
            "mic": "XTSE",
            "listing_country": "CA",
            "listing_ticker": ticker,
            "listing_resolution_method": "CA_CAD_CONTEXT",
            "listing_resolution_confidence": 0.80,
        })

    # US-listed security. Currency USD plus CUSIP is useful evidence, but does
    # not distinguish NYSE from NASDAQ. Preserve US listing country while
    # leaving exchange unresolved.
    if (
        pd.notna(currency)
        and str(currency) == "USD"
        and pd.notna(
            row.get(
                "cusip_normalised"
            )
        )
        and pd.notna(ticker)
        and re.fullmatch(
            r"[A-Z][A-Z0-9\.\-]{0,9}",
            str(ticker),
        )
    ):
        return pd.Series({
            "exchange": pd.NA,
            "mic": pd.NA,
            "listing_country": "US",
            "listing_ticker": ticker,
            "listing_resolution_method": "US_LISTING_COUNTRY_ONLY",
            "listing_resolution_confidence": 0.70,
        })

    return pd.Series({
        "exchange": pd.NA,
        "mic": pd.NA,
        "listing_country": (
            listing_country_source
            if pd.notna(
                listing_country_source
            )
            else pd.NA
        ),
        "listing_ticker": ticker,
        "listing_resolution_method": "UNRESOLVED",
        "listing_resolution_confidence": 0.00,
    })


listing_resolution_df = (
    security_seed_identified_df.apply(
        infer_listing_from_observation,
        axis=1,
    )
)

# Block 1 may already contain source-native columns named exchange, mic,
# listing_country or share_class. Their normalised values are preserved in
# source_* columns. Remove the raw names before adding canonical resolved
# listing fields so row indexing always returns one scalar rather than a
# duplicate-column Series.
CANONICAL_LISTING_OUTPUT_COLUMNS = [
    "exchange",
    "mic",
    "listing_country",
    "listing_ticker",
    "listing_resolution_method",
    "listing_resolution_confidence",
    "share_class",
]

security_seed_identity_base_df = (
    security_seed_identified_df.drop(
        columns=[
            column
            for column in CANONICAL_LISTING_OUTPUT_COLUMNS
            if column in security_seed_identified_df.columns
        ],
        errors="ignore",
    )
    .copy()
)

security_seed_listed_df = pd.concat(
    [
        security_seed_identity_base_df.reset_index(drop=True),
        listing_resolution_df.reset_index(drop=True),
    ],
    axis=1,
)

duplicate_listing_columns = (
    security_seed_listed_df.columns[
        security_seed_listed_df.columns.duplicated()
    ]
    .tolist()
)

if duplicate_listing_columns:
    raise RuntimeError(
        "Duplicate columns remain after listing resolution: "
        f"{duplicate_listing_columns}"
    )

security_seed_listed_df[
    "share_class"
] = security_seed_listed_df.apply(
    lambda row: (
        row[
            "source_share_class"
        ]
        if pd.notna(
            row[
                "source_share_class"
            ]
        )
        else classify_share_class(
            row[
                "listing_ticker"
            ],
            row[
                "exchange"
            ],
        )
    ),
    axis=1,
)


print(
    "Canonical listing output columns:",
    [
        column
        for column in CANONICAL_LISTING_OUTPUT_COLUMNS
        if column in security_seed_listed_df.columns
    ],
)

print(
    "Duplicate columns after listing resolution:",
    int(
        security_seed_listed_df.columns.duplicated().sum()
    ),
)


def build_issuer_master(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    grouped = (
        dataframe
        .groupby(
            "issuer_id",
            dropna=False,
        )
        .agg(
            issuer_name=(
                "issuer_name",
                mode_or_first,
            ),
            issuer_name_normalised=(
                "issuer_name_normalised",
                mode_or_first,
            ),
            lei=(
                "lei_normalised",
                mode_or_first,
            ),
            issuer_country=(
                "issuer_country_normalised",
                mode_or_first,
            ),
            issuer_identity_type=(
                "issuer_identity_type",
                mode_or_first,
            ),
            issuer_identity_value=(
                "issuer_identity_value",
                mode_or_first,
            ),
            first_observed_snapshot_date=(
                "snapshot_date",
                "min",
            ),
            last_observed_snapshot_date=(
                "snapshot_date",
                "max",
            ),
            first_public_available_date=(
                "available_date",
                "min",
            ),
            last_public_available_date=(
                "available_date",
                "max",
            ),
            observation_count=(
                "issuer_id",
                "size",
            ),
            etf_count=(
                "etf",
                "nunique",
            ),
        )
        .reset_index()
    )

    grouped[
        "issuer_status"
    ] = "PROVISIONAL"

    grouped[
        "source_system"
    ] = "SEC_NPORT"

    grouped[
        "schema_version"
    ] = SECURITY_MASTER_SCHEMA_VERSION

    return grouped


def choose_listing_resolution(
    group: pd.DataFrame,
) -> pd.Series:
    ordered = (
        group.sort_values(
            [
                "listing_resolution_confidence",
                "available_date",
                "snapshot_date",
            ],
            ascending=[
                False,
                False,
                False,
            ],
            na_position="last",
        )
    )

    resolved = ordered[
        ordered[
            "listing_resolution_method"
        ].ne(
            "UNRESOLVED"
        )
    ]

    selected = (
        resolved.iloc[0]
        if not resolved.empty
        else ordered.iloc[0]
    )

    return pd.Series({
        "ticker": mode_or_first(
            group[
                "ticker_normalised"
            ]
        ),
        "listing_ticker": selected[
            "listing_ticker"
        ],
        "exchange": selected[
            "exchange"
        ],
        "mic": selected[
            "mic"
        ],
        "listing_country": selected[
            "listing_country"
        ],
        "share_class": selected[
            "share_class"
        ],
        "listing_resolution_method": selected[
            "listing_resolution_method"
        ],
        "listing_resolution_confidence": selected[
            "listing_resolution_confidence"
        ],
        "listing_evidence_count": int(
            group[
                "listing_resolution_method"
            ]
            .ne(
                "UNRESOLVED"
            )
            .sum()
        ),
        "listing_exchange_variant_count": int(
            group[
                "exchange"
            ]
            .dropna()
            .nunique()
        ),
    })


def build_security_master(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    base = (
        dataframe
        .groupby(
            "security_id",
            dropna=False,
        )
        .agg(
            issuer_id=(
                "issuer_id",
                mode_or_first,
            ),
            security_name=(
                "security_name",
                mode_or_first,
            ),
            issuer_name=(
                "issuer_name",
                mode_or_first,
            ),
            isin=(
                "isin_normalised",
                mode_or_first,
            ),
            cusip=(
                "cusip_normalised",
                mode_or_first,
            ),
            lei=(
                "lei_normalised",
                mode_or_first,
            ),
            issuer_country=(
                "issuer_country_normalised",
                mode_or_first,
            ),
            currency=(
                "currency_normalised",
                mode_or_first,
            ),
            security_identity_type=(
                "security_identity_type",
                mode_or_first,
            ),
            security_identity_value=(
                "security_identity_value",
                mode_or_first,
            ),
            first_observed_snapshot_date=(
                "snapshot_date",
                "min",
            ),
            last_observed_snapshot_date=(
                "snapshot_date",
                "max",
            ),
            first_public_available_date=(
                "available_date",
                "min",
            ),
            last_public_available_date=(
                "available_date",
                "max",
            ),
            observation_count=(
                "security_id",
                "size",
            ),
            etf_count=(
                "etf",
                "nunique",
            ),
        )
        .reset_index()
    )

    listing = (
        dataframe.groupby(
            "security_id",
            dropna=False,
            group_keys=False,
        )
        .apply(
            choose_listing_resolution,
            include_groups=False,
        )
        .reset_index()
    )

    grouped = base.merge(
        listing,
        on="security_id",
        how="left",
        validate="1:1",
    )

    grouped[
        "country"
    ] = grouped[
        "issuer_country"
    ]

    grouped[
        "security_type"
    ] = "EQUITY_OR_EQUITY_LINKED"

    grouped[
        "listing_status"
    ] = "OBSERVED_IN_ETF_HOLDINGS"

    grouped[
        "mapping_status"
    ] = np.where(
        grouped[
            "security_identity_type"
        ].isin(
            [
                "ISIN",
                "CUSIP",
            ]
        ),
        "IDENTIFIER_BASED",
        "PROVISIONAL_FUZZY_KEY",
    )

    grouped[
        "listing_mapping_status"
    ] = np.select(
        [
            grouped[
                "listing_resolution_confidence"
            ].ge(0.90),
            grouped[
                "listing_resolution_confidence"
            ].ge(0.70),
        ],
        [
            "HIGH_CONFIDENCE",
            "PROVISIONAL",
        ],
        default="UNRESOLVED",
    )

    grouped[
        "source_system"
    ] = "SEC_NPORT"

    grouped[
        "schema_version"
    ] = SECURITY_MASTER_SCHEMA_VERSION

    preferred_order = [
        "security_id",
        "issuer_id",
        "security_name",
        "issuer_name",
        "ticker",
        "listing_ticker",
        "isin",
        "cusip",
        "lei",
        "issuer_country",
        "country",
        "listing_country",
        "exchange",
        "mic",
        "currency",
        "share_class",
        "security_type",
        "listing_status",
        "mapping_status",
        "listing_mapping_status",
        "security_identity_type",
        "security_identity_value",
        "listing_resolution_method",
        "listing_resolution_confidence",
        "listing_evidence_count",
        "listing_exchange_variant_count",
        "first_observed_snapshot_date",
        "last_observed_snapshot_date",
        "first_public_available_date",
        "last_public_available_date",
        "observation_count",
        "etf_count",
        "source_system",
        "schema_version",
    ]

    return grouped[
        [
            column
            for column
            in preferred_order
            if column in grouped.columns
        ]
    ]


issuer_master_df = build_issuer_master(
    security_seed_listed_df
)

security_master_df = build_security_master(
    security_seed_listed_df
)

print(
    f"Issuer master: {len(issuer_master_df):,} provisional issuers"
)

print(
    f"Security master: {len(security_master_df):,} provisional securities"
)

print(
    "Securities with resolved exchanges:",
    int(
        security_master_df[
            "exchange"
        ].notna().sum()
    ),
)

display(
    security_master_df.head()
)

Canonical listing output columns: ['exchange', 'mic', 'listing_country', 'listing_ticker', 'listing_resolution_method', 'listing_resolution_confidence', 'share_class']
Duplicate columns after listing resolution: 0
Issuer master: 566 provisional issuers
Security master: 512 provisional securities
Securities with resolved exchanges: 71


,security_id,issuer_id,security_name,issuer_name,ticker,listing_ticker,isin,cusip,lei,issuer_country,country,listing_country,exchange,mic,currency,share_class,security_type,listing_status,mapping_status,listing_mapping_status,security_identity_type,security_identity_value,listing_resolution_method,listing_resolution_confidence,listing_evidence_count,listing_exchange_variant_count,first_observed_snapshot_date,last_observed_snapshot_date,first_public_available_date,last_public_available_date,observation_count,etf_count,source_system,schema_version
0,GAS_000EFF8BBFFA00B37D9E,GAI_D8F28DDEBA5977F9D69A,XIAMEN TUNGSTEN CO LTD-A COMMON STOCK,"Xiamen Tungsten Co., Ltd.",<NA>,<NA>,CNE000001D15,<NA>,300300SEC2FOC4PL5N49,CN,CN,<NA>,<NA>,<NA>,CNY,<NA>,EQUITY_OR_EQUITY_LINKED,OBSERVED_IN_ETF_HOLDINGS,IDENTIFIER_BASED,UNRESOLVED,ISIN,CNE000001D15,UNRESOLVED,0.0,0,0,2021-06-30,2026-03-31,2021-08-30,2026-05-29,20,1,SEC_NPORT,1.2.0
1,GAS_012FF9156F0DFDA56EDC,GAI_B4458F7BDC67FFDB71E4,TI Fluid Systems PLC,TI Fluid Systems PLC,<NA>,<NA>,GB00BYQB9V88,000000000,5493001T9RXVD6OAWY46,GB,GB,<NA>,<NA>,<NA>,GBP,<NA>,EQUITY_OR_EQUITY_LINKED,OBSERVED_IN_ETF_HOLDINGS,IDENTIFIER_BASED,UNRESOLVED,ISIN,GB00BYQB9V88,UNRESOLVED,0.0,0,0,2022-01-31,2022-04-30,2022-03-29,2022-06-27,2,1,SEC_NPORT,1.2.0
2,GAS_018F78AA0262BAA2EF41,GAI_CAE0EE7F2C53BE3D2D56,Merdeka Copper Gold Tbk PT,Merdeka Copper Gold Tbk PT,MDKA,MDKA,ID1000134406,000000000,894500DF5TJXCOE26L58,ID,ID,<NA>,<NA>,<NA>,IDR,<NA>,EQUITY_OR_EQUITY_LINKED,OBSERVED_IN_ETF_HOLDINGS,IDENTIFIER_BASED,UNRESOLVED,ISIN,ID1000134406,UNRESOLVED,0.0,0,0,2023-09-29,2026-03-31,2023-11-21,2026-05-21,10,1,SEC_NPORT,1.2.0
3,GAS_01F546D04C0C4F72A262,GAI_5CD5347816CF9C46E768,Aston Martin Lagonda Global Holdings PLC,Aston Martin Lagonda Global Ho,AML,AML,GB00BFXZC448,000000000,213800167WOVOK5ZC776,GB,GB,<NA>,<NA>,<NA>,GBP,<NA>,EQUITY_OR_EQUITY_LINKED,OBSERVED_IN_ETF_HOLDINGS,IDENTIFIER_BASED,UNRESOLVED,ISIN,GB00BFXZC448,UNRESOLVED,0.0,0,0,2019-09-30,2020-09-30,2019-11-19,2020-11-23,5,1,SEC_NPORT,1.2.0
4,GAS_021CBE637C40EA92408A,GAI_2F165C93319FA20296BA,MaxLinear Inc,MaxLinear Inc,MXL,MXL,US57776J1007,57776J100,549300EMOI0SM2IY4F64,US,US,US,<NA>,<NA>,USD,<NA>,EQUITY_OR_EQUITY_LINKED,OBSERVED_IN_ETF_HOLDINGS,IDENTIFIER_BASED,PROVISIONAL,ISIN,US57776J1007,US_LISTING_COUNTRY_ONLY,0.7,17,0,2019-09-30,2026-03-31,2019-11-27,2026-05-21,20,2,SEC_NPORT,1.2.0


In [ ]:
# 8. IDENTIFIER, NAME, LISTING AND SOURCE-MAPPING HISTORY
# ------------------------------------------------

IDENTIFIER_COLUMNS = {
    "ISIN": "isin_normalised",
    "CUSIP": "cusip_normalised",
    "LEI": "lei_normalised",
    "TICKER": "ticker_normalised",
}


def build_identifier_history(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    records = []

    for identifier_type, column in (
        IDENTIFIER_COLUMNS.items()
    ):
        if column not in dataframe.columns:
            continue

        subset = dataframe.loc[
            dataframe[
                column
            ].notna()
        ].copy()

        for row in subset.itertuples(
            index=False
        ):
            records.append({
                "security_id": getattr(
                    row,
                    "security_id",
                ),
                "issuer_id": getattr(
                    row,
                    "issuer_id",
                ),
                "identifier_type": identifier_type,
                "identifier_value": getattr(
                    row,
                    column,
                ),
                "valid_from": getattr(
                    row,
                    "available_date",
                    pd.NaT,
                ),
                "valid_to": pd.NaT,
                "observed_snapshot_date": getattr(
                    row,
                    "snapshot_date",
                    pd.NaT,
                ),
                "source_etf": getattr(
                    row,
                    "etf",
                    pd.NA,
                ),
                "source_accession_number": getattr(
                    row,
                    "accession_number",
                    pd.NA,
                ),
                "source_system": "SEC_NPORT",
                "is_primary": (
                    identifier_type
                    in {
                        "ISIN",
                        "CUSIP",
                    }
                ),
            })

    history = pd.DataFrame(
        records
    )

    if history.empty:
        return pd.DataFrame(
            columns=[
                "security_id",
                "issuer_id",
                "identifier_type",
                "identifier_value",
                "valid_from",
                "valid_to",
                "observed_snapshot_date",
                "source_etf",
                "source_accession_number",
                "source_system",
                "is_primary",
            ]
        )

    return (
        history.sort_values(
            [
                "security_id",
                "identifier_type",
                "identifier_value",
                "valid_from",
                "observed_snapshot_date",
            ]
        )
        .drop_duplicates(
            [
                "security_id",
                "identifier_type",
                "identifier_value",
                "valid_from",
                "source_accession_number",
            ]
        )
        .reset_index(drop=True)
    )


def build_name_history(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    columns = [
        "security_id",
        "issuer_id",
        "issuer_name",
        "issuer_name_normalised",
        "available_date",
        "snapshot_date",
        "etf",
        "accession_number",
    ]

    available_columns = [
        column
        for column in columns
        if column in dataframe.columns
    ]

    history = dataframe[
        available_columns
    ].copy()

    history = history.rename(
        columns={
            "available_date": "valid_from",
            "snapshot_date": "observed_snapshot_date",
            "etf": "source_etf",
            "accession_number": "source_accession_number",
        }
    )

    history[
        "valid_to"
    ] = pd.NaT

    history[
        "source_system"
    ] = "SEC_NPORT"

    return (
        history.sort_values(
            [
                "security_id",
                "valid_from",
                "observed_snapshot_date",
            ]
        )
        .drop_duplicates(
            [
                "security_id",
                "issuer_name_normalised",
                "valid_from",
                "source_accession_number",
            ]
        )
        .reset_index(drop=True)
    )


def build_listing_history(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    columns = [
        "security_id",
        "issuer_id",
        "ticker_normalised",
        "listing_ticker",
        "exchange",
        "mic",
        "issuer_country_normalised",
        "listing_country",
        "currency_normalised",
        "share_class",
        "listing_resolution_method",
        "listing_resolution_confidence",
        "available_date",
        "snapshot_date",
        "next_available_date",
        "etf",
        "accession_number",
    ]

    history = dataframe[
        [
            column
            for column in columns
            if column in dataframe.columns
        ]
    ].copy()

    history = history.rename(
        columns={
            "ticker_normalised": "source_ticker",
            "issuer_country_normalised": "issuer_country",
            "currency_normalised": "currency",
            "available_date": "valid_from",
            "next_available_date": "valid_to",
            "snapshot_date": "observed_snapshot_date",
            "etf": "source_etf",
            "accession_number": "source_accession_number",
        }
    )

    if "valid_to" not in history.columns:
        history[
            "valid_to"
        ] = pd.NaT

    history[
        "source_system"
    ] = "SEC_NPORT"

    return (
        history.sort_values(
            [
                "security_id",
                "valid_from",
                "observed_snapshot_date",
            ]
        )
        .drop_duplicates(
            [
                "security_id",
                "listing_ticker",
                "exchange",
                "mic",
                "valid_from",
                "source_accession_number",
            ]
        )
        .reset_index(drop=True)
    )


def build_source_security_bridge(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    bridge_columns = [
        "security_id",
        "issuer_id",
        "etf",
        "series_id",
        "series_name",
        "accession_number",
        "holding_id",
        "snapshot_date",
        "filing_date",
        "available_date",
        "ticker",
        "ticker_normalised",
        "listing_ticker",
        "isin",
        "isin_normalised",
        "cusip",
        "cusip_normalised",
        "lei",
        "lei_normalised",
        "issuer_name",
        "issuer_name_normalised",
        "issuer_country_normalised",
        "currency_normalised",
        "exchange",
        "mic",
        "listing_country",
        "share_class",
        "listing_resolution_method",
        "listing_resolution_confidence",
        "security_identity_type",
        "security_identity_value",
        "issuer_identity_type",
        "issuer_identity_value",
    ]

    result = dataframe[
        [
            column
            for column
            in bridge_columns
            if column in dataframe.columns
        ]
    ].copy()

    result[
        "source_system"
    ] = "SEC_NPORT"

    result[
        "mapping_method"
    ] = result[
        "security_identity_type"
    ]

    result[
        "mapping_confidence"
    ] = (
        result[
            "security_identity_type"
        ]
        .map({
            "ISIN": 1.00,
            "CUSIP": 0.95,
            "ISSUER_TICKER_CURRENCY": 0.70,
            "ISSUER_SECURITY_NAME": 0.55,
        })
        .fillna(0.25)
    )

    return (
        result.drop_duplicates()
        .reset_index(drop=True)
    )


security_identifier_history_df = (
    build_identifier_history(
        security_seed_listed_df
    )
)

security_name_history_df = (
    build_name_history(
        security_seed_listed_df
    )
)

security_listing_history_df = (
    build_listing_history(
        security_seed_listed_df
    )
)

source_security_bridge_df = (
    build_source_security_bridge(
        security_seed_listed_df
    )
)

print(
    f"Identifier-history rows: {len(security_identifier_history_df):,}"
)

print(
    f"Name-history rows: {len(security_name_history_df):,}"
)

print(
    f"Listing-history rows: {len(security_listing_history_df):,}"
)

print(
    f"Source bridge rows: {len(source_security_bridge_df):,}"
)

Identifier-history rows: 22,368
Name-history rows: 7,958
Listing-history rows: 7,940
Source bridge rows: 7,968


In [ ]:
# 9. ATTACH SECURITY IDS AND LISTING METADATA TO ETF INTERVALS
# ------------------------------------------------


def attach_security_ids_to_intervals(
    intervals: pd.DataFrame,
    source_bridge: pd.DataFrame,
    master: pd.DataFrame,
) -> pd.DataFrame:
    """
    Attach canonical IDs and canonical listing fields to Block 1 intervals.
    """

    left = (
        canonicalise_block_1_columns(
            intervals
        )
        .copy()
    )

    stale_columns = [
        "security_id",
        "issuer_id",
        "security_id_x",
        "security_id_y",
        "issuer_id_x",
        "issuer_id_y",
        "mapping_method",
        "mapping_confidence",
        "exchange",
        "mic",
        "listing_country",
        "share_class",
        "listing_resolution_method",
        "listing_resolution_confidence",
    ]

    left = left.drop(
        columns=[
            column
            for column
            in stale_columns
            if column in left.columns
        ],
        errors="ignore",
    )

    deterministic = (
        standardise_security_seed(
            left
        )
    )

    deterministic = (
        assign_identity_keys(
            deterministic
        )
    )

    deterministic[
        "mapping_method"
    ] = deterministic[
        "security_identity_type"
    ]

    deterministic[
        "mapping_confidence"
    ] = (
        deterministic[
            "security_identity_type"
        ]
        .map({
            "ISIN": 1.00,
            "CUSIP": 0.95,
            "ISSUER_TICKER_CURRENCY": 0.70,
            "ISSUER_SECURITY_NAME": 0.55,
        })
        .fillna(0.25)
    )

    right = (
        canonicalise_block_1_columns(
            source_bridge
        )
        .copy()
    )

    exact_keys = [
        key
        for key in [
            "accession_number",
            "holding_id",
        ]
        if (
            key in deterministic.columns
            and key in right.columns
        )
    ]

    if exact_keys == [
        "accession_number",
        "holding_id",
    ]:
        exact_bridge = (
            right[
                exact_keys
                + [
                    "security_id",
                    "issuer_id",
                    "mapping_method",
                    "mapping_confidence",
                ]
            ]
            .dropna(
                subset=exact_keys
            )
            .sort_values(
                exact_keys
            )
            .drop_duplicates(
                exact_keys,
                keep="last",
            )
            .rename(
                columns={
                    "security_id": "bridge_security_id",
                    "issuer_id": "bridge_issuer_id",
                    "mapping_method": "bridge_mapping_method",
                    "mapping_confidence": "bridge_mapping_confidence",
                }
            )
        )

        result = deterministic.merge(
            exact_bridge,
            on=exact_keys,
            how="left",
            validate="m:1",
        )

        exact_match = result[
            "bridge_security_id"
        ].notna()

        result.loc[
            exact_match,
            "security_id",
        ] = result.loc[
            exact_match,
            "bridge_security_id",
        ]

        result.loc[
            exact_match,
            "issuer_id",
        ] = result.loc[
            exact_match,
            "bridge_issuer_id",
        ]

        result.loc[
            exact_match,
            "mapping_method",
        ] = result.loc[
            exact_match,
            "bridge_mapping_method",
        ]

        result.loc[
            exact_match,
            "mapping_confidence",
        ] = result.loc[
            exact_match,
            "bridge_mapping_confidence",
        ]

        result = result.drop(
            columns=[
                "bridge_security_id",
                "bridge_issuer_id",
                "bridge_mapping_method",
                "bridge_mapping_confidence",
            ],
            errors="ignore",
        )
    else:
        result = deterministic

    # Preserve interval-level source observations under explicit source_*
    # names. The canonical listing fields merged from security_master_df must
    # remain unsuffixed because all downstream blocks expect those exact names.
    interval_listing_rename_map = {
        "exchange": "interval_source_exchange",
        "mic": "interval_source_mic",
        "listing_country": "interval_source_listing_country",
        "share_class": "interval_source_share_class",
        "listing_ticker": "interval_source_listing_ticker",
        "listing_resolution_method": (
            "interval_source_listing_resolution_method"
        ),
        "listing_resolution_confidence": (
            "interval_source_listing_resolution_confidence"
        ),
        "listing_mapping_status": (
            "interval_source_listing_mapping_status"
        ),
    }

    result = result.rename(
        columns={
            source: target
            for source, target in interval_listing_rename_map.items()
            if source in result.columns
        }
    )

    listing_columns = [
        "security_id",
        "listing_ticker",
        "exchange",
        "mic",
        "listing_country",
        "share_class",
        "listing_resolution_method",
        "listing_resolution_confidence",
        "listing_mapping_status",
    ]

    canonical_listing_bridge = (
        master[
            listing_columns
        ]
        .drop_duplicates(
            "security_id"
        )
        .copy()
    )

    result = result.merge(
        canonical_listing_bridge,
        on="security_id",
        how="left",
        validate="m:1",
    )

    duplicate_columns_after_listing_merge = (
        result.columns[
            result.columns.duplicated()
        ]
        .tolist()
    )

    if duplicate_columns_after_listing_merge:
        raise RuntimeError(
            "Duplicate columns remain after the Step 9 listing merge: "
            f"{duplicate_columns_after_listing_merge}"
        )

    print(
        "Canonical listing columns after Step 9 merge:",
        [
            column
            for column in [
                "listing_ticker",
                "exchange",
                "mic",
                "listing_country",
                "share_class",
                "listing_resolution_method",
                "listing_resolution_confidence",
                "listing_mapping_status",
            ]
            if column in result.columns
        ],
    )

    print(
        "Interval source-listing columns preserved:",
        [
            column
            for column in result.columns
            if column.startswith("interval_source_")
        ],
    )

    required_output_columns = {
        "security_id",
        "issuer_id",
        "mapping_method",
        "mapping_confidence",
        "exchange",
        "listing_country",
    }

    missing = (
        required_output_columns
        .difference(
            result.columns
        )
    )

    if missing:
        raise RuntimeError(
            "Step 9 failed to create required columns: "
            f"{sorted(missing)}"
        )

    result[
        "membership_valid_from"
    ] = pd.to_datetime(
        result[
            "available_date"
        ],
        errors="coerce",
    )

    result[
        "membership_valid_to"
    ] = pd.to_datetime(
        result[
            "next_available_date"
        ],
        errors="coerce",
    )

    result[
        "source_system"
    ] = "SEC_NPORT"

    return result


security_etf_membership_intervals_df = (
    attach_security_ids_to_intervals(
        etf_constituent_intervals_df,
        source_security_bridge_df,
        security_master_df,
    )
)

mapping_rate = (
    security_etf_membership_intervals_df[
        "security_id"
    ]
    .notna()
    .mean()
)

listing_rate = (
    security_etf_membership_intervals_df[
        "exchange"
    ]
    .notna()
    .mean()
)

print(
    "Membership intervals with security IDs:",
    f"{mapping_rate:.2%}",
)

print(
    "Membership intervals with resolved exchanges:",
    f"{listing_rate:.2%}",
)

display(
    security_etf_membership_intervals_df.head()
)

Canonical listing columns after Step 9 merge: ['listing_ticker', 'exchange', 'mic', 'listing_country', 'share_class', 'listing_resolution_method', 'listing_resolution_confidence', 'listing_mapping_status']
Interval source-listing columns preserved: ['interval_source_exchange', 'interval_source_mic', 'interval_source_listing_country', 'interval_source_share_class']
Membership intervals with security IDs: 100.00%
Membership intervals with resolved exchanges: 22.30%


,etf,series_id,series_name,snapshot_date,filing_date,effective_date,accession_number,ticker,cusip,isin,issuer_name,security_name,lei,other_identifier,other_identifier_description,country,currency,asset_category,issuer_type,market_value_usd,market_value,weight,balance,is_constituent,available_date,next_available_date,is_latest_public_snapshot,interval_source_exchange,interval_source_mic,interval_source_listing_country,interval_source_share_class,holding_id,security_name_normalised,issuer_name_normalised,ticker_normalised,isin_normalised,cusip_normalised,lei_normalised,issuer_country_normalised,currency_normalised,source_exchange_normalised,source_mic_normalised,source_listing_country_normalised,source_share_class,isin_country,source_system,source_module,security_master_schema_version,issuer_identity_type,issuer_identity_value,security_identity_type,security_identity_value,issuer_id,security_id,mapping_method,mapping_confidence,listing_ticker,exchange,mic,listing_country,share_class,listing_resolution_method,listing_resolution_confidence,listing_mapping_status,membership_valid_from,membership_valid_to
0,CARZ,S000032974,First Trust NASDAQ Global Auto Index Fund,2019-09-30,2019-11-19,2019-11-19,0001752724-19-167449,7267,000000000,JP3854600008,Honda Motor Co Ltd,Honda Motor Co Ltd,549300P7ZYCQJ36CCS16,<NA>,<NA>,JP,JPY,EC,CORP,1458067.03,1458067.03,0.082736,56335.0,1,2019-11-19,2020-02-26,False,<NA>,<NA>,<NA>,<NA>,<NA>,HONDA MOTOR CO LTD,HONDA MOTOR CO LTD,7267,JP3854600008,000000000,549300P7ZYCQJ36CCS16,JP,JPY,<NA>,<NA>,<NA>,<NA>,JP,SEC_NPORT,BLOCK_1_ETF_HOLDINGS,1.2.0,LEI,549300P7ZYCQJ36CCS16,ISIN,JP3854600008,GAI_A117AEB09F9E52CA9C52,GAS_ED6AFE1A9088E8B4AE2C,ISIN,1.0,7267,TSE,XTKS,JP,<NA>,JP_CONTEXT_NUMERIC_TICKER,0.92,HIGH_CONFIDENCE,2019-11-19,2020-02-26
1,CARZ,S000032974,First Trust NASDAQ Global Auto Index Fund,2019-09-30,2019-11-19,2019-11-19,0001752724-19-167449,DAI,000000000,DE0007100000,Daimler AG,Daimler AG,529900R27DL06UVNT076,<NA>,<NA>,DE,EUR,EC,CORP,1416230.33,1416230.33,0.080362,28482.0,1,2019-11-19,2020-02-26,False,<NA>,<NA>,<NA>,<NA>,<NA>,DAIMLER AG,DAIMLER AG,DAI,DE0007100000,000000000,529900R27DL06UVNT076,DE,EUR,<NA>,<NA>,<NA>,<NA>,DE,SEC_NPORT,BLOCK_1_ETF_HOLDINGS,1.2.0,LEI,529900R27DL06UVNT076,ISIN,DE0007100000,GAI_596BC65ADE0A1130CA97,GAS_A5E6ADBC02B825859E4D,ISIN,1.0,MBG,<NA>,<NA>,<NA>,<NA>,UNRESOLVED,0.00,UNRESOLVED,2019-11-19,2020-02-26
2,CARZ,S000032974,First Trust NASDAQ Global Auto Index Fund,2019-09-30,2019-11-19,2019-11-19,0001752724-19-167449,7203,000000000,JP3633400001,Toyota Motor Corp,Toyota Motor Corp,5493006W3QUS5LMH6R84,<NA>,<NA>,JP,JPY,EC,CORP,1366518.53,1366518.53,0.077541,20476.0,1,2019-11-19,2020-02-26,False,<NA>,<NA>,<NA>,<NA>,<NA>,TOYOTA MOTOR CORP,TOYOTA MOTOR CORP,7203,JP3633400001,000000000,5493006W3QUS5LMH6R84,JP,JPY,<NA>,<NA>,<NA>,<NA>,JP,SEC_NPORT,BLOCK_1_ETF_HOLDINGS,1.2.0,LEI,5493006W3QUS5LMH6R84,ISIN,JP3633400001,GAI_00675E60A2AE275E9299,GAS_EE9BD66DEE8D2A470436,ISIN,1.0,7203,TSE,XTKS,JP,<NA>,JP_CONTEXT_NUMERIC_TICKER,0.92,HIGH_CONFIDENCE,2019-11-19,2020-02-26
3,CARZ,S000032974,First Trust NASDAQ Global Auto Index Fund,2019-09-30,2019-11-19,2019-11-19,0001752724-19-167449,GM,37045V100,US37045V1008,General Motors Co,General Motors Co,54930070NSV60J38I987,<NA>,<NA>,US,USD,EC,CORP,1353552.72,1353552.72,0.076805,36114.0,1,2019-11-19,2020-02-26,False,<NA>,<NA>,<NA>,<NA>,<NA>,GENERAL MOTORS CO,GENERAL MOTORS CO,GM,US37045V1008,37045V100,54930070NSV60J38I987,US,USD,<NA>,<NA>,<NA>,<NA>,US,SEC_NPORT,BLOCK_1_ETF_HOLDINGS,1.2.0,LEI,54930070NSV60J38I987,ISIN,US37045V1008,GAI_79133399BDD82B2B04AF,GAS_86E56DD2866215516E5B,ISIN,1.0,GM,<NA>,<NA>,US,<NA>,US_LISTING_COUNTRY_ONLY,0.70,PROVISIONAL,2019-11-19,2020-02-26
4,CARZ,S000032974,First Trust NASDAQ Global Auto Index Fund,2019-09-30,2019-11-19,2019-11-19,0001752724-19-167449,F,345370860,US3453708600,Ford Motor Co,Ford Motor Co,20S05OYHG0MQM4VUIC57,<NA>,<NA>,US,USD,EC,CORP,1338019.52,1338019.52,0.075924,146072.0,1,2019-11-19,2020-02-26,False,<NA

In [ ]:
# 10. COLLISION, LISTING AND MAPPING-QUALITY DIAGNOSTICS
# ------------------------------------------------


def build_identifier_collision_report(
    identifier_history: pd.DataFrame,
) -> pd.DataFrame:
    if identifier_history.empty:
        return pd.DataFrame(
            columns=[
                "identifier_type",
                "identifier_value",
                "security_count",
                "issuer_count",
                "securities",
                "issuers",
                "first_observed",
                "last_observed",
            ]
        )

    report = (
        identifier_history
        .groupby(
            [
                "identifier_type",
                "identifier_value",
            ],
            dropna=False,
        )
        .agg(
            security_count=(
                "security_id",
                "nunique",
            ),
            issuer_count=(
                "issuer_id",
                "nunique",
            ),
            securities=(
                "security_id",
                lambda values: tuple(
                    sorted(
                        set(
                            values.dropna()
                        )
                    )
                ),
            ),
            issuers=(
                "issuer_id",
                lambda values: tuple(
                    sorted(
                        set(
                            values.dropna()
                        )
                    )
                ),
            ),
            first_observed=(
                "valid_from",
                "min",
            ),
            last_observed=(
                "valid_from",
                "max",
            ),
        )
        .reset_index()
    )

    return (
        report.loc[
            (
                report[
                    "security_count"
                ] > 1
            )
            | (
                report[
                    "issuer_count"
                ] > 1
            )
        ]
        .sort_values(
            [
                "security_count",
                "issuer_count",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )


def build_listing_resolution_report(
    master: pd.DataFrame,
) -> pd.DataFrame:
    return (
        master.groupby(
            [
                "listing_resolution_method",
                "listing_mapping_status",
                "exchange",
                "listing_country",
            ],
            dropna=False,
        )
        .agg(
            security_count=(
                "security_id",
                "nunique",
            ),
            issuer_count=(
                "issuer_id",
                "nunique",
            ),
            average_confidence=(
                "listing_resolution_confidence",
                "mean",
            ),
        )
        .reset_index()
        .sort_values(
            [
                "security_count",
                "average_confidence",
            ],
            ascending=[
                False,
                False,
            ],
        )
        .reset_index(drop=True)
    )


def build_security_master_quality_report(
    master: pd.DataFrame,
    source_bridge: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    total = len(
        master
    )

    rows.append({
        "metric": "security_count",
        "value": total,
        "rate": np.nan,
    })

    coverage_columns = [
        "isin",
        "cusip",
        "lei",
        "ticker",
        "issuer_country",
        "currency",
        "listing_country",
        "exchange",
        "mic",
        "share_class",
    ]

    for column in coverage_columns:
        count = (
            int(
                master[
                    column
                ].notna().sum()
            )
            if column in master.columns
            else 0
        )

        rows.append({
            "metric": f"securities_with_{column}",
            "value": count,
            "rate": (
                count / total
                if total
                else np.nan
            ),
        })

    identifier_based = int(
        master[
            "mapping_status"
        ].eq(
            "IDENTIFIER_BASED"
        ).sum()
    )

    rows.append({
        "metric": "identifier_based_security_ids",
        "value": identifier_based,
        "rate": (
            identifier_based / total
            if total
            else np.nan
        ),
    })

    high_confidence_listing = int(
        master[
            "listing_mapping_status"
        ].eq(
            "HIGH_CONFIDENCE"
        ).sum()
    )

    rows.append({
        "metric": "high_confidence_listing_ids",
        "value": high_confidence_listing,
        "rate": (
            high_confidence_listing / total
            if total
            else np.nan
        ),
    })

    unresolved_listing = int(
        master[
            "listing_mapping_status"
        ].eq(
            "UNRESOLVED"
        ).sum()
    )

    rows.append({
        "metric": "unresolved_listing_ids",
        "value": unresolved_listing,
        "rate": (
            unresolved_listing / total
            if total
            else np.nan
        ),
    })

    low_confidence_source_rows = int(
        source_bridge[
            "mapping_confidence"
        ].lt(
            0.80
        ).sum()
    )

    rows.append({
        "metric": "low_confidence_source_rows",
        "value": low_confidence_source_rows,
        "rate": (
            low_confidence_source_rows
            / len(source_bridge)
            if len(source_bridge)
            else np.nan
        ),
    })

    return pd.DataFrame(
        rows
    )


identifier_collision_report_df = (
    build_identifier_collision_report(
        security_identifier_history_df
    )
)

security_listing_resolution_report_df = (
    build_listing_resolution_report(
        security_master_df
    )
)

security_master_quality_df = (
    build_security_master_quality_report(
        security_master_df,
        source_security_bridge_df,
    )
)

security_master_unresolved_df = (
    source_security_bridge_df.loc[
        source_security_bridge_df[
            "mapping_confidence"
        ].lt(
            0.80
        )
    ]
    .sort_values(
        [
            "mapping_confidence",
            "issuer_name",
            "ticker_normalised",
        ]
    )
    .reset_index(drop=True)
)

security_listing_unresolved_df = (
    security_master_df.loc[
        security_master_df[
            "listing_mapping_status"
        ].eq(
            "UNRESOLVED"
        )
    ]
    .sort_values(
        [
            "issuer_country",
            "issuer_name",
            "ticker",
        ]
    )
    .reset_index(drop=True)
)

security_listing_conflict_df = (
    security_master_df.loc[
        security_master_df[
            "listing_exchange_variant_count"
        ].gt(
            1
        )
    ]
    .sort_values(
        [
            "listing_exchange_variant_count",
            "issuer_name",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

display(
    security_master_quality_df
)

display(
    security_listing_resolution_report_df.head(100)
)

,metric,value,rate
0,security_count,512,NaN
1,securities_with_isin,454,0.886719
2,securities_with_cusip,356,0.695312
3,securities_with_lei,352,0.687500
4,securities_with_ticker,201,0.392578
5,securities_with_issuer_country,512,1.000000
6,securities_with_currency,512,1.000000
7,securities_with_listing_country,172,0.335938
8,securities_with_exchange,71,0.138672
9,securities_with_mic,71,0.138672


,listing_resolution_method,listing_mapping_status,exchange,listing_country,security_count,issuer_count,average_confidence
0,UNRESOLVED,UNRESOLVED,NaN,NaN,340,304,0.00
1,US_LISTING_COUNTRY_ONLY,PROVISIONAL,NaN,US,101,99,0.70
2,HKD_CONTEXT_NUMERIC_TICKER,HIGH_CONFIDENCE,HKEX,HK,18,18,0.92
3,JP_CONTEXT_NUMERIC_TICKER,HIGH_CONFIDENCE,TSE,JP,18,18,0.92
4,CA_CAD_CONTEXT,PROVISIONAL,TSX,CA,13,11,0.80
5,KR_CONTEXT_NUMERIC_TICKER,HIGH_CONFIDENCE,KRX,KR,11,11,0.90
6,AU_CONTEXT_TICKER,PROVISIONAL,ASX,AU,7,7,0.85
7,TW_CONTEXT_NUMERIC_TICKER,HIGH_CONFIDENCE,TWSE,TW,3,3,0.90
8,CN_CONTEXT_NUMERIC_TICKER,HIGH_CONFIDENCE,SZSE,CN,1,1,0.93


In [ ]:
# 11. POINT-IN-TIME SECURITY-MASTER ACCESS FUNCTIONS
# ------------------------------------------------


def get_etf_universe_as_of(
    membership_intervals: pd.DataFrame,
    as_of_date: str | pd.Timestamp,
    etfs: str | Iterable[str] | None = None,
) -> pd.DataFrame:
    """
    Return ETF constituents publicly available at a specified model date.
    """

    date = pd.Timestamp(
        as_of_date
    )

    frame = (
        membership_intervals.copy()
    )

    valid = (
        (
            frame[
                "membership_valid_from"
            ] <= date
        )
        & (
            frame[
                "membership_valid_to"
            ].isna()
            | (
                date
                < frame[
                    "membership_valid_to"
                ]
            )
        )
    )

    frame = frame.loc[
        valid
    ].copy()

    if etfs is not None:
        etf_list = (
            [etfs]
            if isinstance(
                etfs,
                str,
            )
            else list(etfs)
        )

        frame = frame.loc[
            frame[
                "etf"
            ].isin(
                etf_list
            )
        ].copy()

    frame[
        "model_date"
    ] = date

    return (
        frame.sort_values(
            [
                "etf",
                "security_id",
            ]
        )
        .reset_index(drop=True)
    )


def get_security_master_as_of(
    master: pd.DataFrame,
    as_of_date: str | pd.Timestamp,
) -> pd.DataFrame:
    """
    Return securities known to the database by the specified public date.
    """

    date = pd.Timestamp(
        as_of_date
    )

    frame = master.loc[
        master[
            "first_public_available_date"
        ] <= date
    ].copy()

    frame[
        "model_date"
    ] = date

    return (
        frame.sort_values(
            "security_id"
        )
        .reset_index(drop=True)
    )


def get_listing_history_as_of(
    listing_history: pd.DataFrame,
    as_of_date: str | pd.Timestamp,
) -> pd.DataFrame:
    """
    Return listing observations publicly available at a specified date.
    """

    date = pd.Timestamp(
        as_of_date
    )

    frame = listing_history.copy()

    valid = (
        (
            frame[
                "valid_from"
            ] <= date
        )
        & (
            frame[
                "valid_to"
            ].isna()
            | (
                date
                < frame[
                    "valid_to"
                ]
            )
        )
    )

    frame = frame.loc[
        valid
    ].copy()

    frame[
        "model_date"
    ] = date

    return (
        frame.sort_values(
            [
                "security_id",
                "exchange",
                "listing_ticker",
            ]
        )
        .reset_index(drop=True)
    )


def lookup_security(
    query: str,
    master: pd.DataFrame = security_master_df,
) -> pd.DataFrame:
    """
    Search IDs, names, tickers, listing venues and major identifiers.
    """

    needle = (
        str(query)
        .strip()
        .upper()
    )

    columns = [
        "security_id",
        "issuer_id",
        "security_name",
        "issuer_name",
        "ticker",
        "listing_ticker",
        "exchange",
        "mic",
        "listing_country",
        "isin",
        "cusip",
        "lei",
    ]

    mask = pd.Series(
        False,
        index=master.index,
    )

    for column in columns:
        if column in master.columns:
            mask |= (
                master[
                    column
                ]
                .astype(
                    "string"
                )
                .str.upper()
                .str.contains(
                    re.escape(
                        needle
                    ),
                    na=False,
                )
            )

    return (
        master.loc[
            mask
        ]
        .copy()
        .reset_index(drop=True)
    )

In [ ]:
# 12. DOWNSTREAM MODULE INTERFACES
# ------------------------------------------------

REGIONAL_ENTITY_BRIDGE_SCHEMA = [
    "source_system",
    "source_entity_id",
    "source_filing_id",
    "filing_date",
    "available_date",
    "issuer_name",
    "lei",
    "isin",
    "ticker",
    "listing_ticker",
    "exchange",
    "mic",
    "issuer_country",
    "listing_country",
    "currency",
    "share_class",
    "security_id",
    "issuer_id",
    "mapping_method",
    "mapping_confidence",
]

lean_security_map_seed_df = (
    security_master_df[
        [
            "security_id",
            "issuer_id",
            "ticker",
            "listing_ticker",
            "exchange",
            "mic",
            "issuer_country",
            "listing_country",
            "currency",
            "share_class",
            "first_public_available_date",
            "last_public_available_date",
            "listing_resolution_method",
            "listing_resolution_confidence",
        ]
    ]
    .copy()
    .rename(
        columns={
            "ticker": "source_ticker",
            "first_public_available_date": "known_from",
            "last_public_available_date": "last_observed",
        }
    )
)

lean_security_map_seed_df[
    "lean_symbol"
] = pd.NA

lean_security_map_seed_df[
    "lean_market"
] = lean_security_map_seed_df[
    "exchange"
].map({
    "NYSE": "usa",
    "NASDAQ": "usa",
    "NYSE_ARCA": "usa",
    "LSE": "uk",
    "TSE": "japan",
    "HKEX": "hongkong",
    "SSE": "china",
    "SZSE": "china",
    "BSE": "china",
    "KRX": "southkorea",
    "KOSDAQ": "southkorea",
    "ASX": "australia",
    "TSX": "canada",
})

lean_security_map_seed_df[
    "valid_from"
] = lean_security_map_seed_df[
    "known_from"
]

lean_security_map_seed_df[
    "valid_to"
] = pd.NaT

lean_security_map_seed_df[
    "mapping_status"
] = np.where(
    lean_security_map_seed_df[
        "exchange"
    ].notna(),
    "LISTING_RESOLVED_SYMBOL_PENDING",
    "LISTING_AND_SYMBOL_PENDING",
)

MAINLAND_CHINA_SECURITY_FILTER = (
    security_master_df[
        "exchange"
    ].isin(
        {
            "SSE",
            "SZSE",
            "BSE",
        }
    )
)

HONG_KONG_SECURITY_FILTER = (
    security_master_df[
        "exchange"
    ].eq(
        "HKEX"
    )
)

JAPAN_SECURITY_FILTER = (
    security_master_df[
        "exchange"
    ].eq(
        "TSE"
    )
)

KOREA_SECURITY_FILTER = (
    security_master_df[
        "exchange"
    ].isin(
        {
            "KRX",
            "KOSDAQ",
        }
    )
)

print(
    "Mainland China securities:",
    int(
        MAINLAND_CHINA_SECURITY_FILTER.sum()
    ),
)

print(
    "Hong Kong securities:",
    int(
        HONG_KONG_SECURITY_FILTER.sum()
    ),
)

print(
    "Japan securities:",
    int(
        JAPAN_SECURITY_FILTER.sum()
    ),
)

print(
    "Korea securities:",
    int(
        KOREA_SECURITY_FILTER.sum()
    ),
)

display(
    lean_security_map_seed_df.head()
)

Mainland China securities: 1
Hong Kong securities: 18
Japan securities: 18
Korea securities: 11


,security_id,issuer_id,source_ticker,listing_ticker,exchange,mic,issuer_country,listing_country,currency,share_class,known_from,last_observed,listing_resolution_method,listing_resolution_confidence,lean_symbol,lean_market,valid_from,valid_to,mapping_status
0,GAS_000EFF8BBFFA00B37D9E,GAI_D8F28DDEBA5977F9D69A,<NA>,<NA>,<NA>,<NA>,CN,<NA>,CNY,<NA>,2021-08-30,2026-05-29,UNRESOLVED,0.0,<NA>,NaN,2021-08-30,NaT,LISTING_AND_SYMBOL_PENDING
1,GAS_012FF9156F0DFDA56EDC,GAI_B4458F7BDC67FFDB71E4,<NA>,<NA>,<NA>,<NA>,GB,<NA>,GBP,<NA>,2022-03-29,2022-06-27,UNRESOLVED,0.0,<NA>,NaN,2022-03-29,NaT,LISTING_AND_SYMBOL_PENDING
2,GAS_018F78AA0262BAA2EF41,GAI_CAE0EE7F2C53BE3D2D56,MDKA,MDKA,<NA>,<NA>,ID,<NA>,IDR,<NA>,2023-11-21,2026-05-21,UNRESOLVED,0.0,<NA>,NaN,2023-11-21,NaT,LISTING_AND_SYMBOL_PENDING
3,GAS_01F546D04C0C4F72A262,GAI_5CD5347816CF9C46E768,AML,AML,<NA>,<NA>,GB,<NA>,GBP,<NA>,2019-11-19,2020-11-23,UNRESOLVED,0.0,<NA>,NaN,2019-11-19,NaT,LISTING_AND_SYMBOL_PENDING
4,GAS_021CBE637C40EA92408A,GAI_2F165C93319FA20296BA,MXL,MXL,<NA>,<NA>,US,US,USD,<NA>,2019-11-27,2026-05-21,US_LISTING_COUNTRY_ONLY,0.7,<NA>,NaN,2019-11-27,NaT,LISTING_AND_SYMBOL_PENDING


In [ ]:
# 13. FINAL BLOCK 2 DATA BUNDLE
# ------------------------------------------------

block_2_data = {
    # Standardised and mapped source observations
    "security_seed_standardised_df": security_seed_standardised_df,
    "security_seed_identified_df": security_seed_identified_df,
    "security_seed_listed_df": security_seed_listed_df,
    "source_security_bridge_df": source_security_bridge_df,

    # Core master tables
    "issuer_master_df": issuer_master_df,
    "security_master_df": security_master_df,
    "security_identifier_history_df": security_identifier_history_df,
    "security_name_history_df": security_name_history_df,
    "security_listing_history_df": security_listing_history_df,

    # Point-in-time ETF relationship tables
    "security_etf_membership_intervals_df": (
        security_etf_membership_intervals_df
    ),

    # Diagnostics
    "identifier_collision_report_df": identifier_collision_report_df,
    "security_master_quality_df": security_master_quality_df,
    "security_master_unresolved_df": security_master_unresolved_df,
    "security_listing_resolution_report_df": (
        security_listing_resolution_report_df
    ),
    "security_listing_unresolved_df": security_listing_unresolved_df,
    "security_listing_conflict_df": security_listing_conflict_df,

    # Downstream integration seeds
    "lean_security_map_seed_df": lean_security_map_seed_df,
}

print(
    "Block 2 transformations complete."
)

print(
    "\nAvailable outputs:"
)

for name, dataframe in (
    block_2_data.items()
):
    print(
        f"  {name}: "
        f"{len(dataframe):,} rows × "
        f"{len(dataframe.columns):,} columns"
    )

Block 2 transformations complete.

Available outputs:
  security_seed_standardised_df: 8,028 rows × 40 columns
  security_seed_identified_df: 8,028 rows × 46 columns
  security_seed_listed_df: 8,028 rows × 49 columns
  source_security_bridge_df: 7,968 rows × 36 columns
  issuer_master_df: 566 rows × 16 columns
  security_master_df: 512 rows × 34 columns
  security_identifier_history_df: 22,368 rows × 11 columns
  security_name_history_df: 7,958 rows × 10 columns
  security_listing_history_df: 7,940 rows × 18 columns
  security_etf_membership_intervals_df: 8,028 rows × 66 columns
  identifier_collision_report_df: 189 rows × 8 columns
  security_master_quality_df: 15 rows × 3 columns
  security_master_unresolved_df: 79 rows × 36 columns
  security_listing_resolution_report_df: 9 rows × 7 columns
  security_listing_unresolved_df: 340 rows × 34 columns
  security_listing_conflict_df: 0 rows × 34 columns
  lean_security_map_seed_df: 512 rows × 19 columns


In [ ]:
# 14. MEMORY-SAFE PERSISTENCE AND VALIDATION
# ------------------------------------------------


def make_parquet_safe(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """
    Return a Parquet-safe shallow copy while preserving native numeric and
    datetime columns.
    """

    output = dataframe.copy(
        deep=False
    )

    for column in output.columns:
        if output[column].dtype != "object":
            continue

        non_missing = (
            output[column]
            .dropna()
        )

        if non_missing.empty:
            continue

        sample = non_missing.head(
            10_000
        )

        if sample.map(
            type
        ).nunique() > 1:
            if output is dataframe:
                output = dataframe.copy(
                    deep=False
                )

            output[column] = (
                output[column]
                .astype(
                    "string"
                )
            )

    return output


def persist_dataframe(
    name: str,
    dataframe: pd.DataFrame,
    output_dir: Path,
    *,
    overwrite: bool = True,
) -> dict:
    output_path = (
        output_dir
        / f"{name}.parquet"
    )

    if (
        output_path.exists()
        and not overwrite
    ):
        raise FileExistsError(
            "Refusing to overwrite existing output: "
            f"{output_path}"
        )

    safe_dataframe = (
        make_parquet_safe(
            dataframe
        )
    )

    table = pa.Table.from_pandas(
        safe_dataframe,
        preserve_index=False,
        safe=False,
    )

    pq.write_table(
        table,
        output_path,
        compression="snappy",
        use_dictionary=True,
        write_statistics=True,
        row_group_size=100_000,
    )

    record = {
        "table_name": name,
        "path": str(output_path),
        "row_count": int(
            len(dataframe)
        ),
        "column_count": int(
            len(dataframe.columns)
        ),
        "columns": list(
            map(
                str,
                dataframe.columns,
            )
        ),
        "file_size_bytes": int(
            output_path.stat().st_size
        ),
        "created_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
    }

    del table

    if safe_dataframe is not dataframe:
        del safe_dataframe

    gc.collect()

    return record


def load_block_2_outputs(
    output_dir: Path = BLOCK_2_OUTPUT_DIR,
) -> dict[str, pd.DataFrame]:
    manifest_path = (
        output_dir
        / "block_2_manifest.json"
    )

    if not manifest_path.exists():
        raise FileNotFoundError(
            f"Block 2 manifest not found: {manifest_path}"
        )

    with manifest_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        manifest = json.load(file)

    loaded = {}

    for table_record in manifest[
        "tables"
    ]:
        table_path = Path(
            table_record[
                "path"
            ]
        )

        if not table_path.exists():
            raise FileNotFoundError(
                "Manifest table is missing: "
                f"{table_path}"
            )

        loaded[
            table_record[
                "table_name"
            ]
        ] = pd.read_parquet(
            table_path
        )

    return loaded


if PERSIST_BLOCK_2_OUTPUTS:
    manifest_rows = []

    preferred_order = [
        "security_master_df",
        "issuer_master_df",
        "security_identifier_history_df",
        "security_listing_history_df",
        "security_etf_membership_intervals_df",
        "source_security_bridge_df",
        "lean_security_map_seed_df",
    ]

    persistence_order = (
        [
            name
            for name in preferred_order
            if name in block_2_data
        ]
        + [
            name
            for name in block_2_data
            if name not in preferred_order
        ]
    )

    for table_name in (
        persistence_order
    ):
        manifest_rows.append(
            persist_dataframe(
                table_name,
                block_2_data[
                    table_name
                ],
                BLOCK_2_OUTPUT_DIR,
                overwrite=OVERWRITE_PERSISTED_OUTPUTS,
            )
        )

    block_2_manifest = {
        "block": 2,
        "block_name": (
            "Global Security Master and listing-identity construction"
        ),
        "schema_version": SECURITY_MASTER_SCHEMA_VERSION,
        "created_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "project_root": str(
            PROJECT_ROOT
        ),
        "input_manifest": str(
            BLOCK_1_MANIFEST_PATH
        ),
        "output_directory": str(
            BLOCK_2_OUTPUT_DIR
        ),
        "listing_resolution_policy": [
            "source exchange or MIC",
            "exact ticker suffix",
            "country, ISIN, currency and ticker structure",
            "explicit unresolved state",
        ],
        "tables": manifest_rows,
    }

    with BLOCK_2_MANIFEST_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            block_2_manifest,
            file,
            indent=2,
            ensure_ascii=False,
        )

    block_2_persistence_report_df = (
        pd.DataFrame(
            manifest_rows
        )
    )

    print(
        "Block 2 outputs persisted successfully."
    )

    print(
        "Manifest:",
        BLOCK_2_MANIFEST_PATH,
    )

    display(
        block_2_persistence_report_df[
            [
                "table_name",
                "row_count",
                "column_count",
                "file_size_bytes",
                "path",
            ]
        ]
    )

    required_downstream_tables = {
        "issuer_master_df",
        "security_master_df",
        "security_identifier_history_df",
        "security_listing_history_df",
        "source_security_bridge_df",
        "security_etf_membership_intervals_df",
        "lean_security_map_seed_df",
        "security_listing_resolution_report_df",
        "security_listing_unresolved_df",
    }

    manifest_names = {
        record[
            "table_name"
        ]
        for record in manifest_rows
    }

    missing_tables = (
        required_downstream_tables
        .difference(
            manifest_names
        )
    )

    if missing_tables:
        raise RuntimeError(
            "Persistence validation failed. Missing tables: "
            f"{sorted(missing_tables)}"
        )

    validation_rows = []

    for table_name in sorted(
        required_downstream_tables
    ):
        table_path = (
            BLOCK_2_OUTPUT_DIR
            / f"{table_name}.parquet"
        )

        parquet_file = pq.ParquetFile(
            table_path
        )

        original_rows = len(
            block_2_data[
                table_name
            ]
        )

        persisted_rows = (
            parquet_file.metadata.num_rows
        )

        if persisted_rows != original_rows:
            raise RuntimeError(
                f"Row-count mismatch for {table_name}: "
                f"{original_rows:,} original versus "
                f"{persisted_rows:,} persisted."
            )

        validation_rows.append({
            "table_name": table_name,
            "original_rows": original_rows,
            "persisted_rows": persisted_rows,
            "persisted_columns": (
                parquet_file.metadata.num_columns
            ),
            "status": "PASSED",
        })

        del parquet_file
        gc.collect()

    block_2_validation_report_df = (
        pd.DataFrame(
            validation_rows
        )
    )

    display(
        block_2_validation_report_df
    )

    print(
        "Persistence validation passed. "
        "Later blocks can load the complete Security Master, listing history "
        "and ETF intervals without rerunning Blocks 1 or 2."
    )

else:
    block_2_persistence_report_df = (
        pd.DataFrame()
    )

    block_2_validation_report_df = (
        pd.DataFrame()
    )

    print(
        "PERSIST_BLOCK_2_OUTPUTS is False. "
        "Outputs remain available only in the current runtime."
    )

Block 2 outputs persisted successfully.
Manifest: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/interim/block_2/block_2_manifest.json


,table_name,row_count,column_count,file_size_bytes,path
0,security_master_df,512,34,91620,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
1,issuer_master_df,566,16,64017,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
2,security_identifier_history_df,22368,11,97230,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
3,security_listing_history_df,7940,18,73769,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
4,security_etf_membership_intervals_df,8028,66,606834,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
5,source_security_bridge_df,7968,36,221407,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
6,lean_security_map_seed_df,512,19,41664,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
7,security_seed_standardised_df,8028,40,198787,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
8,security_seed_identified_df,8028,46,275148,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
9,security_seed_listed_df,8028,49,285377,/content/drive/MyDrive/Colab Notebooks/00 A1 A...


,table_name,original_rows,persisted_rows,persisted_columns,status
0,issuer_master_df,566,566,16,PASSED
1,lean_security_map_seed_df,512,512,19,PASSED
2,security_etf_membership_intervals_df,8028,8028,66,PASSED
3,security_identifier_history_df,22368,22368,11,PASSED
4,security_listing_history_df,7940,7940,18,PASSED
5,security_listing_resolution_report_df,9,9,7,PASSED
6,security_listing_unresolved_df,340,340,34,PASSED
7,security_master_df,512,512,34,PASSED
8,source_security_bridge_df,7968,7968,36,PASSED


Persistence validation passed. Later blocks can load the complete Security Master, listing history and ETF intervals without rerunning Blocks 1 or 2.


## What Block 2 resolves—and what remains provisional

### Resolved in this block

- Loads revised Block 1 outputs from Parquet without executing Block 1.
- Creates deterministic provisional `issuer_id` and `security_id` values.
- Separates issuer identity from listed-security identity.
- Preserves identifier, name and listing histories with source provenance.
- Resolves exchange, MIC, listing country and share class where source evidence
  is sufficient.
- Attaches IDs and canonical listing metadata to point-in-time ETF intervals.
- Produces collision, low-confidence, listing-conflict and unresolved-listing
  diagnostics.
- Creates common regional and LEAN-facing interfaces.
- Persists all outputs under `data/interim/block_2/`.

### Listing resolution

The resolver does not assume that issuer domicile equals listing venue.

Examples:

- a Chinese issuer with a six-digit Mainland ticker may resolve to SSE, SZSE
  or BSE;
- a Chinese issuer with an HKD-denominated numeric security may resolve to
  HKEX;
- a Chinese issuer with a USD-denominated alphabetic security may be identified
  as US-listed while NYSE versus NASDAQ remains unresolved;
- Japanese and Korean numeric tickers are classified using their own country or
  ISIN context rather than being mistaken for Chinese securities.

### Intentionally provisional

N-PORT holdings cannot always distinguish:

- NYSE from NASDAQ where no exchange code is present;
- depositary receipts from ordinary shares;
- historical exchange migrations;
- all secondary-listing relationships;
- every delisted or renamed security.

These cases remain visible in:

```python
security_listing_unresolved_df
security_listing_conflict_df
security_listing_resolution_report_df
```

No ambiguous listing is silently accepted.

### Downstream execution

Later blocks should load:

```text
data/interim/block_2/block_2_manifest.json
```

and the required Parquet tables directly. They should not execute Block 2.